# FHS Factor Market Simulator V9.7 (point-in-time, hardened gates)


## V9.7 methodology contract

V9.7 keeps the V9 engine (regime-conditioned factor FHS through the v8 calibration
stack) and fixes the gate design plus known bugs:

- Kupiec/Hill gates now compare like with like: an UNCONDITIONAL simulation book vs
  all validation days, and the crisis BASE book vs crisis-regime days. The stress
  ladder book is never Kupiec-gated (it is deliberately severe); it is coverage-gated.
- Cholesky alpha sweep selects the correlation-transport strength on TRAIN data only
  (candidate corr vs train target, benchmarked against a train-fitted Gaussian).
  Validation never touches the choice; `cholesky_uses_validation` stays honestly False.
- Walk-forward refits (2019/2021/2022): stress book must cover the worst realized
  window; the unconditional base book must pass Kupiec on the following year.
- Promotion allows a statistical tie (2-sigma on multi-seed MMD) against naive FHS and
  the same-stack Gaussian ONLY if the candidate's crisis-tail CVaR is strictly closer
  to the eval reference — tails break ties, nothing else does.
- PIT membership CSV is REQUIRED by default (survivorship fallback must be opted into).
- Fixed: undefined `synthetic_factor_norm` at artifact save; stress factor bank now
  exports genuinely stressed paths instead of a copy of the base bank.


## 0. Colab Runtime

Recommended runtime:
- GPU: A100 if available; T4/L4 works for smaller universes.
- Runtime type: Python 3 with GPU.

Default settings are conservative enough for Colab. Increase `N_ASSETS`, `N_EPOCHS`, and `N_SCENARIOS` only after the first end-to-end run succeeds.
        


In [ ]:
#@title Install dependencies
import sys, subprocess, os, textwrap, json, math, random, time

def pip_install(*packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])

pip_install("numpy", "pandas", "scikit-learn", "matplotlib", "tqdm", "requests", "hmmlearn", "yfinance")


In [ ]:
#@title Imports and device
import os, json, math, time, random, warnings, re, gc
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Optional, Tuple
import getpass
import requests

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

try:
    from hmmlearn.hmm import GaussianHMM
    HMM_AVAILABLE = True
except Exception as exc:
    HMM_AVAILABLE = False
    GaussianHMM = None
    print("hmmlearn unavailable; regime detector will fall back to KMeans:", exc)

try:
    import yfinance as yf
    YFINANCE_AVAILABLE = True
except Exception as exc:
    YFINANCE_AVAILABLE = False
    yf = None
    print("yfinance unavailable; macro/VIX conditioning will use internal market features only:", exc)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision("high")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0))


In [ ]:
#@title Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/blsprime_ddpm_market_sim")
DATA_DIR = PROJECT_DIR / "data"
ARTIFACT_DIR = PROJECT_DIR / "artifacts"
CHECKPOINT_DIR = PROJECT_DIR / "checkpoints"

for path in [DATA_DIR, ARTIFACT_DIR, CHECKPOINT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("Project dir:", PROJECT_DIR)
print("Put your CSV in:", DATA_DIR)



## 1. Configuration

Input options:
- `DATA_SOURCE = "fmp"` downloads historical EOD prices from Financial Modeling Prep into Drive.
- `DATA_SOURCE = "csv"` loads a CSV from Drive.
- `DATA_SOURCE = "demo"` creates a synthetic demo dataset.

For CSV:
- first column is date or index;
- remaining columns are either adjusted prices or returns.

If `DATA_MODE = "prices"`, the notebook computes log returns.
If `DATA_MODE = "returns"`, the notebook uses the numeric columns directly.
        


In [ ]:
#@title User configuration (V9.7 FHS)
DATA_SOURCE = "fmp"  #@param ["fmp", "csv", "demo"]
CSV_FILENAME = "market_prices_or_returns.csv"  #@param {type:"string"}
CSV_PATH = DATA_DIR / CSV_FILENAME
DATA_MODE = "prices"  #@param ["prices", "returns"]

# FMP source. The PIT cell below replaces FMP_SYMBOLS with the point-in-time union.
FMP_SYMBOLS = "AUTO_SP500"  #@param {type:"string"}
FMP_START_DATE = "2010-01-01"  #@param {type:"string"}
FMP_END_DATE = ""  # blank = today
FMP_SLEEP_SECONDS = 0.12

# Point-in-time universe (survivorship fix). Upload the membership CSV to Drive DATA_DIR.
USE_PIT_UNIVERSE = True
PIT_MEMBERSHIP_CSV_FILENAME = "sp500_constituents_history.csv"
REQUIRE_PIT_MEMBERSHIP = True
ALLOW_SURVIVORSHIP_FALLBACK = False
PIT_MAX_SYMBOLS = 600
PIT_MIN_MEMBER_DAYS = 252

# Universe / sample construction
N_ASSETS = 500         #@param {type:"integer"}
WINDOW_SIZE = 30       #@param {type:"integer"}
N_REGIMES = 4          #@param {type:"integer"}
VALIDATION_MODE = "walk_forward_crisis"  #@param ["walk_forward_crisis", "temporal_80_20"]
TRAIN_FRACTION = 0.80
WALK_FORWARD_TRAIN_END = "2019-12-31"
MIN_HISTORY_FRACTION = 0.70
REGIME_METHOD = "hmm"  #@param ["hmm", "kmeans"]
USE_MACRO_CONDITIONING = True

# Factor engine (same spine as v8)
MODEL_FAMILY = "fhs_factor_engine_v9_7_conditional_var"
FACTOR_PCA_COMPONENTS = 32
FACTOR_RIDGE_ALPHA = 1.0e-3
RESIDUAL_COV_SHRINKAGE = 0.25
RESIDUAL_BOOTSTRAP_SCALE = 1.00
STRESS_SCENARIO_MODE = True
STRESS_STRATIFIED_SAMPLING = True
STRESS_MIX_WEIGHTS = "0.63,0.22,0.10,0.05"
STRESS_MIX_MULTIPLIERS = "1.0,1.45,2.4,6.0"
# V9.5: stress must move the distribution location, not only the variance.
# These are train-only factor-space shifts; they do not use validation returns.
STRESS_LOCATION_SHIFT_ALPHA = 0.55
STRESS_LOCATION_SHIFT_Q = 0.005
STRESS_LOCATION_SHIFT_MIN_MULTIPLIER = 1.0
STRESS_LOCATION_SHIFT_MAX_DAILY_ABS = 0.006
# V9.5: sparse catastrophe sleeve for endpoint coverage. Calibrated from train-only
# daily equal-weight tail; no validation dates or realized crisis minima are used.
STRESS_CATASTROPHE_SLEEVE = True
STRESS_CATASTROPHE_WEIGHT = 0.020
STRESS_CATASTROPHE_DAILY_Q = 0.001
STRESS_CATASTROPHE_SHOCK_SCALE = 1.15
STRESS_CATASTROPHE_MIN_DAYS = 4
STRESS_CATASTROPHE_MAX_DAYS = 8
STRESS_CATASTROPHE_MIN_DAILY_ABS = 0.025
STRESS_CATASTROPHE_MAX_DAILY_ABS = 0.055
DOWNSIDE_ASYMMETRY = 1.35
FACTOR_CALIBRATION_ALPHA = 0.45
FACTOR_CALIBRATION_SHRINKAGE = 0.20
# V9.5: PIT made the target tail harsher; anchor the lower market-factor tail in train space.
FACTOR_TAIL_LOCATION_ALPHA = 0.50
FACTOR_TAIL_ANCHOR_QUANTILE = 0.01
FACTOR_TAIL_ANCHOR_MAX_DAILY_SHIFT = 0.004
CORR_PROJECTION_DIM = 128

# V9 FHS engine
FHS_EWMA_LAMBDA = 0.94
FHS_BLOCK_LENGTH = 5
FHS_MIN_REGIME_POOL = 120
CALIBRATION_TARGET_WINDOWS = 1024
N_UNCOND_SCENARIOS = 2500
TIE_SIGMA = 2.0

# Dataset plumbing kept from v8 (loaders exist; no neural training in v9)
BATCH_SIZE = 96
NUM_WORKERS = 0
USE_REGIME_BALANCED_SAMPLER = True
REGIME_SAMPLER_POWER = 0.70
REGIME_LOSS_POWER = 0.50
TARGET_REGIME_LOSS_BOOST = 1.75

# Sampling / calibration
N_SCENARIOS = 5000
APPLY_CHOLESKY_CALIBRATION = True
CHOLESKY_CALIBRATION_ALPHA = 0.40
CHOLESKY_ALPHA_SWEEP_VALUES = "0.40,0.50,0.60,0.70,0.80"
CHOLESKY_SWEEP_SUBSAMPLE = 800
CHOLESKY_SHRINKAGE = 0.10
# V9.5: choose the lowest-damage alpha among train-correlation-passing values.
CHOLESKY_SELECTION_OBJECTIVE = "pareto_corr_mmd_cvar"
CHOLESKY_CVAR_OBJECTIVE_WEIGHT = 0.35
# V9.7: train-only daily VaR tail safety margin for non-stress gate books.
# The target is derived from train tail shape, then frozen before validation.
APPLY_DAILY_VAR5_SAFETY_MARGIN = True
DAILY_VAR5_SAFETY_METHOD = "train_tail_shape_q05_minus_half_q01_gap"
DAILY_VAR5_SAFETY_TAIL_Q = 0.01
DAILY_VAR5_SAFETY_STEEPNESS_SHARE = 0.50
DAILY_VAR5_SAFETY_MIN_MULTIPLIER = 1.00
DAILY_VAR5_SAFETY_MAX_MULTIPLIER = 1.80
DAILY_VAR5_SAFETY_MAX_SHIFT = 0.020
TARGET_REGIME = 2
AUTO_TARGET_CRISIS_REGIME = True

# V9.7: daily VaR is validated as a conditional rolling risk model, not as a static scenario book.
CONDITIONAL_VAR_BACKTEST = True
CONDITIONAL_VAR_EWMA_LAMBDA = 0.94
CONDITIONAL_VAR_QUANTILES = "0.05,0.01"
CHRISTOFFERSEN_MIN_P = 0.05

# Evaluation
BASELINE_SCENARIOS = 2500
MMD_PROJECTION_DIM = 128
MMD_MAX_WINDOWS = 1500
MMD_STABILITY_SEEDS = "11,23,37,53,71"
EVAL_SYNTHETIC_SUBSAMPLE_SEEDS = "101,203,307,409,503"
PRIMARY_EVAL_SUBSAMPLE_SEED = 101
CORR_MAE_NEAR_GAUSSIAN_TOL = 0.005
MMD_RATIO_MAX_RESEARCH = 1.05

# V8 hardening kept
RETURN_CLIP_MODE = "bad_print_only"  #@param ["none", "bad_print_only", "wide_quantile"]
RETURN_CLIP_LOWER_Q = 0.0001
RETURN_CLIP_UPPER_Q = 0.9999
BAD_PRINT_ABS_RETURN_LIMIT = 0.80
EVAL_WINDOW_STRIDE = WINDOW_SIZE
STRESS_SHOCK_MODE = "sparse_distribution_shift"
SEVERE_SHOCK_MIN_DAYS = 2
SEVERE_SHOCK_MAX_DAYS = 5
SEVERE_MARKET_LOCATION_SHIFT_Q = 0.01
SEVERE_MARKET_SCALE_BOOST = 1.35
BASELINE_T_COPULA_DF = 5
PORTFOLIO_TEST_COUNT = 32
PORTFOLIO_CONCENTRATION_LEVELS = "0.25,0.40,0.60"
EXPORT_FACTOR_SCENARIO_BANK = True
FACTOR_BANK_DTYPE = "float16"
FACTOR_BANK_REGIME_NAME = "crisis"
SURVIVORSHIP_DISCLOSURE = "point_in_time_membership_with_data_coverage_stat"

# V9 extra gates
WALK_FORWARD_CUTOFFS = "2019-12-31,2021-12-31,2022-12-31"
WALK_FORWARD_EVAL_DAYS = 252
WALK_FORWARD_REFIT_SCENARIOS = 2000
KUPIEC_MIN_P = 0.05
HILL_REL_TOL = 0.50
PER_ASSET_Q05_TOL = 0.35

# Endpoint / research deployment contract
ENDPOINT_DEFAULT_SCENARIOS = 5000
ENDPOINT_MIN_SCENARIOS = 2000
ENDPOINT_MIN_STRESS_SCENARIOS = 5000
ENDPOINT_STRESS_QUANTILE = 0.01
NORMAL_CANDIDATE_MODEL = "fhs_v9_base"
STRESS_CANDIDATE_MODEL = "fhs_v9_stress"
PRODUCT_CANDIDATE_MODEL = STRESS_CANDIDATE_MODEL
ENDPOINT_SMALL_REQUEST_POLICY = "reject_or_aggregate"

CONFIG = dict(
    seed=SEED, data_source=DATA_SOURCE, data_mode=DATA_MODE, fmp_symbols=FMP_SYMBOLS,
    fmp_start_date=FMP_START_DATE, fmp_end_date=FMP_END_DATE, csv_filename=CSV_FILENAME,
    use_pit_universe=USE_PIT_UNIVERSE, pit_membership_csv=PIT_MEMBERSHIP_CSV_FILENAME,
    pit_max_symbols=PIT_MAX_SYMBOLS, pit_min_member_days=PIT_MIN_MEMBER_DAYS,
    n_assets=N_ASSETS, window_size=WINDOW_SIZE, min_history_fraction=MIN_HISTORY_FRACTION,
    validation_mode=VALIDATION_MODE, train_fraction=TRAIN_FRACTION,
    walk_forward_train_end=WALK_FORWARD_TRAIN_END,
    regime_method=REGIME_METHOD, use_macro_conditioning=USE_MACRO_CONDITIONING,
    n_regimes=N_REGIMES,
    model_family=MODEL_FAMILY, factor_pca_components=FACTOR_PCA_COMPONENTS,
    factor_ridge_alpha=FACTOR_RIDGE_ALPHA, residual_cov_shrinkage=RESIDUAL_COV_SHRINKAGE,
    residual_bootstrap_scale=RESIDUAL_BOOTSTRAP_SCALE, stress_scenario_mode=STRESS_SCENARIO_MODE,
    stress_stratified_sampling=STRESS_STRATIFIED_SAMPLING,
    stress_mix_weights=STRESS_MIX_WEIGHTS, stress_mix_multipliers=STRESS_MIX_MULTIPLIERS,
    stress_location_shift_alpha=STRESS_LOCATION_SHIFT_ALPHA,
    stress_location_shift_q=STRESS_LOCATION_SHIFT_Q,
    stress_location_shift_min_multiplier=STRESS_LOCATION_SHIFT_MIN_MULTIPLIER,
    stress_location_shift_max_daily_abs=STRESS_LOCATION_SHIFT_MAX_DAILY_ABS,
    stress_catastrophe_sleeve=STRESS_CATASTROPHE_SLEEVE,
    stress_catastrophe_weight=STRESS_CATASTROPHE_WEIGHT,
    stress_catastrophe_daily_q=STRESS_CATASTROPHE_DAILY_Q,
    stress_catastrophe_shock_scale=STRESS_CATASTROPHE_SHOCK_SCALE,
    stress_catastrophe_min_days=STRESS_CATASTROPHE_MIN_DAYS,
    stress_catastrophe_max_days=STRESS_CATASTROPHE_MAX_DAYS,
    stress_catastrophe_min_daily_abs=STRESS_CATASTROPHE_MIN_DAILY_ABS,
    stress_catastrophe_max_daily_abs=STRESS_CATASTROPHE_MAX_DAILY_ABS,
    downside_asymmetry=DOWNSIDE_ASYMMETRY, factor_calibration_alpha=FACTOR_CALIBRATION_ALPHA,
    factor_calibration_shrinkage=FACTOR_CALIBRATION_SHRINKAGE,
    factor_tail_location_alpha=FACTOR_TAIL_LOCATION_ALPHA,
    factor_tail_anchor_quantile=FACTOR_TAIL_ANCHOR_QUANTILE,
    factor_tail_anchor_max_daily_shift=FACTOR_TAIL_ANCHOR_MAX_DAILY_SHIFT,
    corr_projection_dim=CORR_PROJECTION_DIM,
    fhs_ewma_lambda=FHS_EWMA_LAMBDA, fhs_block_length=FHS_BLOCK_LENGTH,
    fhs_min_regime_pool=FHS_MIN_REGIME_POOL,
    n_scenarios=N_SCENARIOS,
    apply_cholesky_calibration=APPLY_CHOLESKY_CALIBRATION,
    cholesky_calibration_alpha=CHOLESKY_CALIBRATION_ALPHA,
    cholesky_shrinkage=CHOLESKY_SHRINKAGE,
    cholesky_alpha_sweep_values=CHOLESKY_ALPHA_SWEEP_VALUES,
    cholesky_selection_objective=CHOLESKY_SELECTION_OBJECTIVE,
    cholesky_cvar_objective_weight=CHOLESKY_CVAR_OBJECTIVE_WEIGHT,
    apply_daily_var5_safety_margin=APPLY_DAILY_VAR5_SAFETY_MARGIN,
    daily_var5_safety_method=DAILY_VAR5_SAFETY_METHOD,
    daily_var5_safety_tail_q=DAILY_VAR5_SAFETY_TAIL_Q,
    daily_var5_safety_steepness_share=DAILY_VAR5_SAFETY_STEEPNESS_SHARE,
    daily_var5_safety_min_multiplier=DAILY_VAR5_SAFETY_MIN_MULTIPLIER,
    daily_var5_safety_max_multiplier=DAILY_VAR5_SAFETY_MAX_MULTIPLIER,
    daily_var5_safety_max_shift=DAILY_VAR5_SAFETY_MAX_SHIFT,
    conditional_var_backtest=CONDITIONAL_VAR_BACKTEST,
    conditional_var_ewma_lambda=CONDITIONAL_VAR_EWMA_LAMBDA,
    conditional_var_quantiles=CONDITIONAL_VAR_QUANTILES,
    christoffersen_min_p=CHRISTOFFERSEN_MIN_P,
    target_regime=TARGET_REGIME, baseline_scenarios=BASELINE_SCENARIOS,
    mmd_projection_dim=MMD_PROJECTION_DIM, mmd_max_windows=MMD_MAX_WINDOWS,
    mmd_stability_seeds=MMD_STABILITY_SEEDS,
    eval_synthetic_subsample_seeds=EVAL_SYNTHETIC_SUBSAMPLE_SEEDS,
    primary_eval_subsample_seed=PRIMARY_EVAL_SUBSAMPLE_SEED,
    corr_mae_near_gaussian_tol=CORR_MAE_NEAR_GAUSSIAN_TOL,
    mmd_ratio_max_research=MMD_RATIO_MAX_RESEARCH,
    return_clip_mode=RETURN_CLIP_MODE,
    return_clip_lower_q=RETURN_CLIP_LOWER_Q,
    return_clip_upper_q=RETURN_CLIP_UPPER_Q,
    bad_print_abs_return_limit=BAD_PRINT_ABS_RETURN_LIMIT,
    eval_window_stride=EVAL_WINDOW_STRIDE,
    stress_shock_mode=STRESS_SHOCK_MODE,
    severe_shock_min_days=SEVERE_SHOCK_MIN_DAYS,
    severe_shock_max_days=SEVERE_SHOCK_MAX_DAYS,
    severe_market_location_shift_q=SEVERE_MARKET_LOCATION_SHIFT_Q,
    severe_market_scale_boost=SEVERE_MARKET_SCALE_BOOST,
    baseline_t_copula_df=BASELINE_T_COPULA_DF,
    portfolio_test_count=PORTFOLIO_TEST_COUNT,
    portfolio_concentration_levels=PORTFOLIO_CONCENTRATION_LEVELS,
    export_factor_scenario_bank=EXPORT_FACTOR_SCENARIO_BANK,
    factor_bank_dtype=FACTOR_BANK_DTYPE,
    factor_bank_regime_name=FACTOR_BANK_REGIME_NAME,
    require_pit_membership=REQUIRE_PIT_MEMBERSHIP,
    allow_survivorship_fallback=ALLOW_SURVIVORSHIP_FALLBACK,
    n_uncond_scenarios=N_UNCOND_SCENARIOS,
    tie_sigma=TIE_SIGMA,
    survivorship_disclosure=SURVIVORSHIP_DISCLOSURE,
    walk_forward_cutoffs=WALK_FORWARD_CUTOFFS,
    walk_forward_eval_days=WALK_FORWARD_EVAL_DAYS,
    walk_forward_refit_scenarios=WALK_FORWARD_REFIT_SCENARIOS,
    kupiec_min_p=KUPIEC_MIN_P, hill_rel_tol=HILL_REL_TOL, per_asset_q05_tol=PER_ASSET_Q05_TOL,
    endpoint_default_scenarios=ENDPOINT_DEFAULT_SCENARIOS,
    endpoint_min_scenarios=ENDPOINT_MIN_SCENARIOS,
    endpoint_min_stress_scenarios=ENDPOINT_MIN_STRESS_SCENARIOS,
    endpoint_stress_quantile=ENDPOINT_STRESS_QUANTILE,
    normal_candidate_model=NORMAL_CANDIDATE_MODEL,
    stress_candidate_model=STRESS_CANDIDATE_MODEL,
    product_candidate_model=PRODUCT_CANDIDATE_MODEL,
    endpoint_small_request_policy=ENDPOINT_SMALL_REQUEST_POLICY,
)
print(json.dumps({k: v for k, v in CONFIG.items() if k != "factor_columns"}, indent=2, default=str))


In [ ]:
#@title Large universe: FMP -> DataHub S&P 500 -> Wikipedia -> static fallback
import io
import re

# Practical defaults:
# - T4/L4: MAX_FMP_SYMBOLS=250-350, BATCH_SIZE=96-128, D_MODEL=160
# - A100: MAX_FMP_SYMBOLS=500, BATCH_SIZE=192, D_MODEL=192
UNIVERSE_SOURCE = "sp500"
MAX_FMP_SYMBOLS = 500
DATAHUB_SP500_URL = "https://raw.githubusercontent.com/datasets/s-and-p-500-companies/main/data/constituents.csv"
STATIC_SP500_SYMBOLS = """MMM AOS ABT ABBV ACN ADBE AMD AES AFL A APD ABNB AKAM ALB ARE ALGN ALLE LNT ALL GOOGL GOOG MO AMZN AMCR AEE AEP AXP AIG AMT AWK AMP AME AMGN APH ADI AON APA APO AAPL AMAT APP APTV ACGL ADM ARES ANET AJG AIZ T ATO ADSK ADP AZO AVB AVY AXON BKR BALL BAC BAX BDX BRK-B BBY TECH BIIB BLK BX XYZ BNY BA BKNG BSX BMY AVGO BR BRO BF-B BLDR BG BXP CHRW CDNS CPT COF CAH CCL CARR CVNA CASY CAT CBOE CBRE CDW COR CNC CNP CF CRL SCHW CHTR CVX CMG CB CHD CIEN CI CINF CTAS CSCO C CFG CLX CME CMS KO CTSH COHR COIN CL CMCSA FIX COP ED STZ CEG COO CPRT GLW CPAY CTVA CSGP COST CRH CRWD CCI CSX CMI CVS DHR DRI DDOG DVA DECK DE DELL DAL DVN DXCM FANG DLR DG DLTR D DPZ DASH DOV DOW DHI DTE DUK DD ETN EBAY ECHO ECL EIX EW EA ELV EME EMR ETR EOG EQT EFX EQIX EQR ERIE ESS EL EG EVRG ES EXC EXE EXPE EXPD EXR XOM FFIV FDS FICO FAST FRT FDX FDXF FIS FITB FSLR FE FISV FLEX F FTNT FTV FOXA FOX BEN FCX GRMN IT GE GEHC GEV GEN GNRC GD GIS GM GPC GILD GPN GL GDDY GS HAL HIG HAS HCA DOC HSIC HSY HPE HLT HD HONA HON HRL HST HWM HPQ HUBB HUM HBAN HII IBM IEX IDXX ITW INCY IR PODD INTC IBKR ICE IFF IP INTU ISRG IVZ INVH IQV IRM JBHT JBL JKHY J JNJ JCI JPM KVUE KDP KEY KEYS KMB KIM KMI KKR KLAC KHC KR LHX LH LRCX LVS LDOS LEN LII LLY LIN LYV LMT L LOW LULU LITE LYB MTB MPC MAR MRSH MLM MRVL MAS MA MKC MCD MCK MDT MRK META MET MTD MGM MCHP MU MSFT MAA MRNA TAP MDLZ MPWR MNST MCO MS MOS MSI MSCI NDAQ NTAP NFLX NEM NWSA NWS NEE NKE NI NDSN NSC NTRS NOC NCLH NRG NUE NVDA NVR NXPI ORLY OXY ODFL OMC ON OKE ORCL OTIS PCAR PKG PLTR PANW PSKY PH PAYX PYPL PNR PEP PFE PCG PM PSX PNW PNC PPG PPL PFG PG PGR PLD PRU PEG PTC PSA PHM PWR QCOM DGX Q RL RJF RTX O REG REGN RF RSG RMD RVTY HOOD ROK ROL ROP ROST RCL SPGI CRM SNDK SBAC SLB STX SRE NOW SHW SPG SWKS SJM SW SNA SOLV SO LUV SWK SBUX STT STLD STE SYK SMCI SYF SNPS SYY TMUS TROW TTWO TPR TRGP TGT TEL TDY TER TSLA TXN TPL TXT TMO TJX TKO TTD TSCO TT TDG TRV TRMB TFC TYL TSN USB UBER UDR ULTA UNP UAL UPS URI UNH UHS VLO VEEV VTR VLTO VRSN VRSK VZ VRTX VRT VTRS VICI V VST VMC WRB GWW WAB WMT DIS WBD WM WAT WEC WFC WELL WST WDC WY WSM WMB WTW WDAY WYNN XEL XYL YUM ZBRA ZBH ZTS""".split()

def sanitize_error_message(exc, api_key=None):
    text = str(exc)
    if api_key:
        text = text.replace(api_key, "***")
    text = re.sub(r"apikey=[^&\s]+", "apikey=***", text)
    return text[:220]

def get_fmp_api_key():
    key = os.environ.get("FMP_API_KEY", "").strip()
    if key:
        return key
    try:
        from google.colab import userdata
        key = (userdata.get("FMP_API_KEY") or "").strip()
        if key:
            os.environ["FMP_API_KEY"] = key
            return key
    except Exception:
        pass
    key = getpass.getpass("Paste FMP API key (input hidden): ").strip()
    os.environ["FMP_API_KEY"] = key
    return key

def clean_symbols(symbols):
    out = []
    for symbol in symbols:
        symbol = str(symbol).upper().strip().replace(".", "-")
        if symbol and re.match(r"^[A-Z0-9\-]+$", symbol):
            out.append(symbol)
    return list(dict.fromkeys(out))

def try_fmp_sp500_constituents():
    api_key = get_fmp_api_key()
    endpoints = [
        ("stable", "https://financialmodelingprep.com/stable/sp500-constituent"),
        ("legacy", "https://financialmodelingprep.com/api/v3/sp500_constituent"),
    ]
    for label, url in endpoints:
        try:
            response = requests.get(url, params={"apikey": api_key}, timeout=30)
            if response.status_code in (401, 402, 403):
                print(f"FMP {label} constituents blocked by plan/key: {response.status_code}")
                continue
            response.raise_for_status()
            payload = response.json()
            if isinstance(payload, list) and payload:
                frame = pd.DataFrame(payload)
                symbol_col = "symbol" if "symbol" in frame.columns else frame.columns[0]
                symbols = clean_symbols(frame[symbol_col].tolist())
                if symbols:
                    print(f"Universe from FMP {label}: {len(symbols)} symbols")
                    return symbols, f"fmp_{label}"
        except Exception as exc:
            print(f"FMP {label} failed:", sanitize_error_message(exc, api_key))
    return [], None

def try_datahub_sp500_constituents():
    try:
        response = requests.get(
            DATAHUB_SP500_URL,
            headers={"User-Agent": "Mozilla/5.0 ddpm-market-simulator"},
            timeout=30,
        )
        response.raise_for_status()
        frame = pd.read_csv(io.StringIO(response.text))
        symbol_col = "Symbol" if "Symbol" in frame.columns else frame.columns[0]
        symbols = clean_symbols(frame[symbol_col].tolist())
        if symbols:
            print(f"Universe from DataHub S&P 500 CSV: {len(symbols)} symbols")
            return symbols, "datahub_sp500"
    except Exception as exc:
        print("DataHub fallback failed:", sanitize_error_message(exc))
    return [], None

def try_wikipedia_sp500_constituents():
    try:
        response = requests.get(
            "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies",
            headers={"User-Agent": "Mozilla/5.0 ddpm-market-simulator"},
            timeout=30,
        )
        response.raise_for_status()
        tables = pd.read_html(io.StringIO(response.text))
        frame = tables[0]
        symbol_col = "Symbol" if "Symbol" in frame.columns else frame.columns[0]
        symbols = clean_symbols(frame[symbol_col].tolist())
        if symbols:
            print(f"Universe from Wikipedia S&P 500: {len(symbols)} symbols")
            return symbols, "wikipedia_sp500"
    except Exception as exc:
        print("Wikipedia fallback failed:", sanitize_error_message(exc))
    return [], None

def fallback_static_sp500_universe():
    symbols = clean_symbols(STATIC_SP500_SYMBOLS)
    print(f"Universe from embedded static S&P 500 fallback: {len(symbols)} symbols")
    return symbols, "embedded_sp500_snapshot"

symbols, resolved_source = [], None
if UNIVERSE_SOURCE == "sp500":
    symbols, resolved_source = try_fmp_sp500_constituents()
    if not symbols:
        symbols, resolved_source = try_datahub_sp500_constituents()
    if not symbols:
        symbols, resolved_source = try_wikipedia_sp500_constituents()
if not symbols:
    symbols, resolved_source = fallback_static_sp500_universe()

selected_symbols = symbols[:MAX_FMP_SYMBOLS]
FMP_SYMBOLS = ",".join(selected_symbols)
DATA_SOURCE = "fmp"
DATA_MODE = "prices"
N_ASSETS = min(N_ASSETS, len(selected_symbols))

CONFIG["fmp_symbols"] = FMP_SYMBOLS
CONFIG["n_assets"] = N_ASSETS
CONFIG["universe_source"] = UNIVERSE_SOURCE
CONFIG["universe_source_resolved"] = resolved_source
CONFIG["max_fmp_symbols"] = MAX_FMP_SYMBOLS
CONFIG["universe_count"] = len(selected_symbols)

print(f"Universe selected: {len(selected_symbols)} stocks from {resolved_source}")
print(selected_symbols[:60])


In [ ]:
#@title Upload PIT S&P 500 membership CSV from your computer to Drive
from google.colab import files
from pathlib import Path
import pandas as pd
import shutil

PROJECT_DIR = Path("/content/drive/MyDrive/blsprime_ddpm_market_sim")
DATA_DIR = PROJECT_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

DEST = DATA_DIR / "sp500_constituents_history.csv"

print("Select this local file:")
print(r"C:\Users\T14 Ultra 7\OneDrive\Escritorio\CT\blsprime-fin\cloud_data\ct\caria_data\sp500_constituents_history.csv")

uploaded = files.upload()

if not uploaded:
    raise RuntimeError("No file uploaded.")

src_name = next(iter(uploaded.keys()))
shutil.move(src_name, DEST)

df = pd.read_csv(DEST)
df.columns = [c.strip().lower() for c in df.columns]

if not {"date", "ticker"}.issubset(df.columns):
    raise ValueError(f"CSV must have columns date,ticker. Found: {df.columns.tolist()}")

df = df[["date", "ticker"]].copy()
df["date"] = pd.to_datetime(df["date"]).dt.strftime("%Y-%m-%d")
df["ticker"] = df["ticker"].astype(str).str.strip().str.upper()
df = df.dropna().drop_duplicates().sort_values(["date", "ticker"])
df.to_csv(DEST, index=False)

print("Saved:", DEST)
print("Rows:", len(df))
print("Tickers:", df["ticker"].nunique())
print("Date range:", df["date"].min(), "to", df["date"].max())
print("OK: V9.5 PIT file is ready.")


In [ ]:
#@title Optional: fetch the PIT membership CSV (best source: your own repo file)
# The canonical source is the repo file cloud_data/ct/caria_data/sp500_constituents_history.csv
# (long format: date,ticker). Upload it to DATA_DIR or /content and skip this cell.
# This cell only tries a public fallback (fja05680 dataset, wide format) if nothing is present.
import io
import requests as _requests

_pit_targets = [DATA_DIR / PIT_MEMBERSHIP_CSV_FILENAME, Path("/content") / PIT_MEMBERSHIP_CSV_FILENAME]
if any(p.exists() for p in _pit_targets):
    print("PIT membership CSV already present; nothing to do.")
else:
    _FALLBACK_URLS = [
        "https://raw.githubusercontent.com/fja05680/sp500/master/S%26P%20500%20Historical%20Components%20%26%20Changes%20(08-17-2024).csv",
        "https://raw.githubusercontent.com/fja05680/sp500/master/S%26P%20500%20Historical%20Components%20%26%20Changes.csv",
    ]
    _text = None
    for _url in _FALLBACK_URLS:
        try:
            _r = _requests.get(_url, timeout=45)
            if _r.ok and len(_r.text) > 1000:
                _text = _r.text
                print("Downloaded fallback:", _url)
                break
            print(f"{_url} -> HTTP {_r.status_code}")
        except Exception as _exc:
            print(f"{_url} -> {_exc}")
    if _text is None:
        print("No fallback available. Upload the repo CSV manually to", DATA_DIR)
    else:
        _wide = pd.read_csv(io.StringIO(_text))
        _wide.columns = [str(c).strip().lower() for c in _wide.columns]
        _date_col = "date" if "date" in _wide.columns else _wide.columns[0]
        _tick_col = "tickers" if "tickers" in _wide.columns else _wide.columns[1]
        _rows = []
        for _, _row in _wide.iterrows():
            _d = pd.to_datetime(_row[_date_col], errors="coerce")
            if pd.isna(_d):
                continue
            for _tk in str(_row[_tick_col]).split(","):
                _tk = _tk.strip().upper()
                if _tk:
                    _rows.append((_d.date().isoformat(), _tk))
        _long = pd.DataFrame(_rows, columns=["date", "ticker"]).drop_duplicates()
        # The fallback is change-dated (sparse); forward-fill to business days.
        _long["date"] = pd.to_datetime(_long["date"])
        _piv = _long.assign(flag=1).pivot_table(index="date", columns="ticker", values="flag", aggfunc="first")
        _piv = _piv.reindex(pd.bdate_range(_piv.index.min(), pd.Timestamp.today())).ffill()
        _out = _piv.stack().reset_index()
        _out.columns = ["date", "ticker", "flag"]
        _out = _out[_out["flag"] > 0][["date", "ticker"]]
        _out["date"] = _out["date"].dt.date
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        _out.to_csv(DATA_DIR / PIT_MEMBERSHIP_CSV_FILENAME, index=False)
        print("Saved normalized long-format membership CSV:", DATA_DIR / PIT_MEMBERSHIP_CSV_FILENAME, len(_out), "rows")


In [ ]:
#@title Point-in-time universe from membership history (survivorship fix)
_pit_candidates = [DATA_DIR / PIT_MEMBERSHIP_CSV_FILENAME, Path("/content") / PIT_MEMBERSHIP_CSV_FILENAME]
PIT_MEMBERSHIP_PATH = next((p for p in _pit_candidates if p.exists()), _pit_candidates[0])
MEMBERSHIP_LONG = None
PIT_UNIVERSE_ACTIVE = False

if USE_PIT_UNIVERSE and REQUIRE_PIT_MEMBERSHIP and not ALLOW_SURVIVORSHIP_FALLBACK and not PIT_MEMBERSHIP_PATH.exists():
    raise RuntimeError(
        "V9.5 requires the point-in-time membership CSV (survivorship gate). "
        f"Upload 'sp500_constituents_history.csv' (repo: cloud_data/ct/caria_data/) to {DATA_DIR} "
        "or /content, or run the fetch cell above. Set ALLOW_SURVIVORSHIP_FALLBACK=True to override."
    )

if USE_PIT_UNIVERSE and PIT_MEMBERSHIP_PATH.exists():
    MEMBERSHIP_LONG = pd.read_csv(PIT_MEMBERSHIP_PATH, parse_dates=["date"])
    MEMBERSHIP_LONG.columns = [c.strip().lower() for c in MEMBERSHIP_LONG.columns]
    MEMBERSHIP_LONG["ticker"] = MEMBERSHIP_LONG["ticker"].astype(str).str.strip().str.upper()
    MEMBERSHIP_LONG = MEMBERSHIP_LONG[MEMBERSHIP_LONG["date"] >= pd.Timestamp(FMP_START_DATE)]
    member_days = MEMBERSHIP_LONG.groupby("ticker").size().sort_values(ascending=False)
    eligible = member_days[member_days >= PIT_MIN_MEMBER_DAYS]
    pit_symbols = list(eligible.index[:PIT_MAX_SYMBOLS])
    if len(pit_symbols) >= 100:
        FMP_SYMBOLS = ",".join(pit_symbols)
        PIT_UNIVERSE_ACTIVE = True
        CONFIG["fmp_symbols"] = f"PIT_UNION_{len(pit_symbols)}"
        CONFIG["pit_universe_active"] = True
        CONFIG["pit_symbols_selected"] = len(pit_symbols)
        CONFIG["pit_membership_rows"] = int(len(MEMBERSHIP_LONG))
        CONFIG["pit_membership_start"] = str(MEMBERSHIP_LONG["date"].min().date())
        CONFIG["pit_membership_end"] = str(MEMBERSHIP_LONG["date"].max().date())
        print(f"PIT universe: {len(pit_symbols)} symbols with >= {PIT_MIN_MEMBER_DAYS} member-days since {FMP_START_DATE}.")
        print("Delisted/exited members are requested too; FMP coverage for them is measured later as pit_member_day_coverage.")
    else:
        print("PIT membership file too thin; falling back to current constituents.")
else:
    if USE_PIT_UNIVERSE:
        print(f"PIT membership CSV not found at {PIT_MEMBERSHIP_PATH}; falling back to current constituents (survivorship-biased, disclosed).")

CONFIG["pit_universe_active"] = bool(PIT_UNIVERSE_ACTIVE)
if not PIT_UNIVERSE_ACTIVE:
    CONFIG["survivorship_disclosure"] = "current_constituents_fallback_survivorship_biased"


In [ ]:
#@title Universe sanity check and sector metadata
selected_symbols = [s.strip().upper() for s in FMP_SYMBOLS.split(",") if s.strip()]
assert selected_symbols, "Empty universe"
assert len(selected_symbols) == len(set(selected_symbols)), "Duplicate symbols in universe"

SECTOR_MAP = {symbol: "Unknown" for symbol in selected_symbols}
try:
    sector_frame = pd.read_csv(DATAHUB_SP500_URL)
    sector_frame["Symbol"] = sector_frame["Symbol"].astype(str).str.upper().str.replace(".", "-", regex=False)
    sector_col = "GICS Sector" if "GICS Sector" in sector_frame.columns else None
    if sector_col:
        SECTOR_MAP.update(dict(zip(sector_frame["Symbol"], sector_frame[sector_col].astype(str))))
except Exception as exc:
    print("Sector metadata fallback: all Unknown:", sanitize_error_message(exc) if "sanitize_error_message" in globals() else str(exc)[:160])

sector_counts = pd.Series([SECTOR_MAP.get(s, "Unknown") for s in selected_symbols], name="sector").value_counts()
print(f"Active FMP universe: {len(selected_symbols)} symbols")
print("First 80 symbols:", selected_symbols[:80])
display(sector_counts.to_frame("symbols"))


## 2. Load Data

The loader is deliberately forgiving:
- detects a date column if present;
- keeps numeric columns;
- drops assets with too many missing observations;
- selects the most liquid/complete assets by variance and coverage;
- builds log returns if given prices.

For FMP, use either:
- Colab secret named `FMP_API_KEY` via the key icon in the left sidebar; or
- paste the key when prompted. It is not written to Drive.
        


In [ ]:
#@title FMP downloader with per-symbol Drive cache
def get_fmp_api_key():
    key = os.environ.get("FMP_API_KEY", "").strip()
    if key:
        return key
    try:
        from google.colab import userdata
        key = (userdata.get("FMP_API_KEY") or "").strip()
        if key:
            os.environ["FMP_API_KEY"] = key
            return key
    except Exception:
        pass
    key = getpass.getpass("Paste FMP API key (input hidden): ").strip()
    os.environ["FMP_API_KEY"] = key
    return key

def sanitize_fmp_error(exc, api_key=None):
    text = str(exc)
    if api_key:
        text = text.replace(api_key, "***")
    text = re.sub(r"apikey=[^&\s]+", "apikey=***", text)
    return text[:300]

def parse_fmp_payload(payload):
    if isinstance(payload, list):
        return payload
    if isinstance(payload, dict) and isinstance(payload.get("historical"), list):
        return payload["historical"]
    if isinstance(payload, dict) and payload.get("Error Message"):
        raise RuntimeError(payload["Error Message"])
    return []

def fetch_fmp_symbol(symbol, api_key, start_date, end_date=None):
    symbol = symbol.strip().upper()
    params = {"symbol": symbol, "from": start_date, "apikey": api_key}
    if end_date:
        params["to"] = end_date

    stable_url = "https://financialmodelingprep.com/stable/historical-price-eod/full"
    response = requests.get(stable_url, params=params, timeout=30)
    rows = parse_fmp_payload(response.json()) if response.ok else []

    if not rows:
        legacy_url = f"https://financialmodelingprep.com/api/v3/historical-price-full/{symbol}"
        legacy_params = {"from": start_date, "apikey": api_key}
        if end_date:
            legacy_params["to"] = end_date
        legacy_response = requests.get(legacy_url, params=legacy_params, timeout=30)
        legacy_response.raise_for_status()
        rows = parse_fmp_payload(legacy_response.json())

    frame = pd.DataFrame(rows)
    if frame.empty:
        raise RuntimeError(f"No FMP rows returned for {symbol}")
    if "date" not in frame.columns:
        raise RuntimeError(f"FMP response for {symbol} has no date column")

    price_col = "adjClose" if "adjClose" in frame.columns else "close"
    if price_col not in frame.columns:
        raise RuntimeError(f"FMP response for {symbol} has no close/adjClose column")

    out = frame[["date", price_col]].copy()
    out["date"] = pd.to_datetime(out["date"], errors="coerce")
    out[price_col] = pd.to_numeric(out[price_col], errors="coerce")
    out = out.dropna().sort_values("date").drop_duplicates("date")
    out = out.rename(columns={price_col: symbol}).set_index("date")
    return out

def download_fmp_prices(symbols, start_date, end_date=None, sleep_seconds=0.12, force_refresh=False):
    api_key = get_fmp_api_key()
    symbol_cache_dir = DATA_DIR / "fmp_symbol_cache"
    symbol_cache_dir.mkdir(parents=True, exist_ok=True)

    frames = []
    errors = {}

    for symbol in tqdm(symbols, desc="FMP download/cache"):
        cache_path = symbol_cache_dir / f"{symbol}_{start_date}_{end_date or 'today'}.csv"
        try:
            if cache_path.exists() and not force_refresh:
                frame = pd.read_csv(cache_path, parse_dates=["date"]).set_index("date")
            else:
                frame = fetch_fmp_symbol(symbol, api_key, start_date, end_date)
                frame.reset_index(names="date").to_csv(cache_path, index=False)
            frames.append(frame)
        except Exception as exc:
            errors[symbol] = sanitize_fmp_error(exc, api_key)
        time.sleep(float(sleep_seconds))

    if not frames:
        raise RuntimeError(f"FMP download returned no usable symbols. Errors: {errors}")

    prices = pd.concat(frames, axis=1).sort_index().ffill(limit=5).dropna(axis=0, how="all")
    min_obs = max(252, int(len(prices) * MIN_HISTORY_FRACTION))
    prices = prices.dropna(axis=1, thresh=min_obs)
    if errors:
        print("Skipped symbols:", errors)
    print(f"Usable price columns: {prices.shape[1]} / requested {len(symbols)}")
    return prices


In [ ]:
#@title Load data with Drive cache: FMP, CSV, or demo
symbol_count = len([s for s in FMP_SYMBOLS.split(",") if s.strip()])
FMP_CACHE_NAME = f"fmp_prices_cache_v2_{FMP_START_DATE}_{FMP_END_DATE or 'today'}_{symbol_count}.csv"
FMP_CACHE_PATH = DATA_DIR / FMP_CACHE_NAME
FORCE_FMP_REFRESH = False  # True only when you intentionally want to redownload.

def make_demo_prices(path: Path, n_days=2400, n_assets=80):
    rng = np.random.default_rng(SEED)
    dates = pd.bdate_range("2015-01-01", periods=n_days)
    market = rng.normal(0.00025, 0.010, size=n_days)
    crisis = rng.random(n_days) < 0.025
    market[crisis] += rng.normal(-0.035, 0.025, size=crisis.sum())
    prices = {}
    for i in range(n_assets):
        beta = rng.uniform(0.6, 1.5)
        sector_shock = rng.normal(0, 0.006, size=n_days)
        idio = rng.normal(0, rng.uniform(0.006, 0.018), size=n_days)
        rets = beta * market + 0.35 * sector_shock + idio
        prices[f"ASSET_{i:03d}"] = 100 * np.exp(np.cumsum(rets))
    demo = pd.DataFrame(prices, index=dates).reset_index(names="date")
    demo.to_csv(path, index=False)
    return path

def load_cached_prices(path):
    frame = pd.read_csv(path)
    frame["date"] = pd.to_datetime(frame["date"], errors="coerce")
    frame = frame.dropna(subset=["date"]).sort_values("date")
    return frame

if DATA_SOURCE == "fmp":
    symbols = [s.strip().upper() for s in FMP_SYMBOLS.split(",") if s.strip()]
    if not symbols:
        raise ValueError("FMP_SYMBOLS is empty")

    if FMP_CACHE_PATH.exists() and not FORCE_FMP_REFRESH:
        print("Using aggregate Drive cache:", FMP_CACHE_PATH)
        raw = load_cached_prices(FMP_CACHE_PATH)
    else:
        print("Building aggregate cache from FMP/per-symbol cache...")
        prices = download_fmp_prices(
            symbols,
            FMP_START_DATE,
            FMP_END_DATE or None,
            FMP_SLEEP_SECONDS,
            force_refresh=FORCE_FMP_REFRESH,
        )
        raw = prices.reset_index(names="date")
        raw.to_csv(FMP_CACHE_PATH, index=False)
        print("Aggregate cache saved:", FMP_CACHE_PATH)

    CSV_PATH = FMP_CACHE_PATH
    DATA_MODE = "prices"

elif DATA_SOURCE == "demo":
    CSV_PATH = make_demo_prices(Path(CSV_PATH))
    raw = pd.read_csv(CSV_PATH)
    DATA_MODE = "prices"

elif DATA_SOURCE == "csv":
    if not Path(CSV_PATH).exists():
        raise FileNotFoundError(f"CSV not found: {CSV_PATH}")
    raw = pd.read_csv(CSV_PATH)

else:
    raise ValueError(f"Unknown DATA_SOURCE: {DATA_SOURCE}")

print(raw.shape)
display(raw.head())


In [ ]:
#@title Prepare return matrix
def prepare_returns(frame: pd.DataFrame, mode: str, n_assets: int) -> pd.DataFrame:
    frame = frame.copy()
    date_col = None
    for col in frame.columns:
        if str(col).lower() in {"date", "datetime", "timestamp"}:
            date_col = col
            break
    if date_col is not None:
        frame[date_col] = pd.to_datetime(frame[date_col], errors="coerce")
        frame = frame.sort_values(date_col).set_index(date_col)
    else:
        frame.index = pd.RangeIndex(len(frame))

    numeric = frame.apply(pd.to_numeric, errors="coerce").sort_index()
    numeric = numeric.ffill(limit=5)
    min_obs = max(252, int(len(numeric) * MIN_HISTORY_FRACTION))
    numeric = numeric.dropna(axis=1, thresh=min_obs)

    if mode == "prices":
        numeric = numeric.loc[:, (numeric > 0).all(axis=0)]
        returns = np.log(numeric).diff()
    else:
        returns = numeric.copy()

    returns = returns.replace([np.inf, -np.inf], np.nan)
    ret_min_obs = max(252, int(len(returns) * MIN_HISTORY_FRACTION))
    returns = returns.dropna(axis=1, thresh=ret_min_obs)

    vol = returns.std(skipna=True)
    valid_cols = vol[(vol > 1e-5) & (vol < 0.25)].index.tolist()
    preferred_order = [s.strip().upper() for s in FMP_SYMBOLS.split(",") if s.strip()]
    ordered = [col for col in preferred_order if col in valid_cols]
    extras = [col for col in valid_cols if col not in ordered]
    keep = (ordered + extras)[:n_assets]

    returns = returns[keep].ffill(limit=5).dropna(axis=0, how="any")

    # V8: preserve real crash tails by default. Only obvious bad prints are neutralized.
    tail_audit = {
        "mode": RETURN_CLIP_MODE,
        "pre_clip_min": float(np.nanmin(returns.values)),
        "pre_clip_max": float(np.nanmax(returns.values)),
        "bad_print_abs_limit": float(BAD_PRINT_ABS_RETURN_LIMIT),
    }
    if RETURN_CLIP_MODE == "wide_quantile":
        lower = returns.quantile(RETURN_CLIP_LOWER_Q)
        upper = returns.quantile(RETURN_CLIP_UPPER_Q)
        returns = returns.clip(lower=lower, upper=upper, axis=1)
        tail_audit["quantile_clip"] = [float(RETURN_CLIP_LOWER_Q), float(RETURN_CLIP_UPPER_Q)]
    elif RETURN_CLIP_MODE == "bad_print_only":
        bad_print_mask = returns.abs() > BAD_PRINT_ABS_RETURN_LIMIT
        tail_audit["bad_print_values_replaced"] = int(bad_print_mask.sum().sum())
        returns = returns.mask(bad_print_mask).ffill(limit=1).dropna(axis=0, how="any")
    elif RETURN_CLIP_MODE != "none":
        raise ValueError(f"Unknown RETURN_CLIP_MODE={RETURN_CLIP_MODE}")
    tail_audit["post_clip_min"] = float(np.nanmin(returns.values))
    tail_audit["post_clip_max"] = float(np.nanmax(returns.values))
    CONFIG["tail_preservation_audit"] = tail_audit
    returns = returns.astype("float32")

    if returns.empty or returns.shape[1] < min(50, n_assets):
        raise RuntimeError(f"Return matrix too small after filtering: {returns.shape}")
    return returns

returns_df = prepare_returns(raw, DATA_MODE, N_ASSETS)
print("Return matrix:", returns_df.shape)
display(returns_df.head())
returns_df.describe().T.head()

print("Tail preservation audit:", json.dumps(CONFIG.get("tail_preservation_audit", {}), indent=2))


In [ ]:
#@title PIT wide panel, membership mask, and coverage stat
USE_PIT_FACTORS = False
PIT_RETURNS_WIDE = None
PIT_MASK = None
PIT_SECTOR_COLUMNS = {}

def _wide_returns_from_raw(frame, mode):
    frame = frame.copy()
    date_col = None
    for col in frame.columns:
        if str(col).lower() in {"date", "datetime", "timestamp"}:
            date_col = col
            break
    if date_col is not None:
        frame[date_col] = pd.to_datetime(frame[date_col], errors="coerce")
        frame = frame.sort_values(date_col).set_index(date_col)
    numeric = frame.apply(pd.to_numeric, errors="coerce").sort_index()
    numeric = numeric.ffill(limit=5)
    if mode == "prices":
        numeric = numeric.where(numeric > 0)
        rets = np.log(numeric).diff()
    else:
        rets = numeric
    rets = rets.replace([np.inf, -np.inf], np.nan)
    rets = rets.where(rets.abs() <= BAD_PRINT_ABS_RETURN_LIMIT)
    return rets.astype("float32")

if PIT_UNIVERSE_ACTIVE and MEMBERSHIP_LONG is not None:
    wide = _wide_returns_from_raw(raw, DATA_MODE)
    keep_cols = [c for c in wide.columns if wide[c].notna().sum() >= 60]
    wide = wide[keep_cols]
    mask = (
        MEMBERSHIP_LONG.assign(flag=True)
        .pivot_table(index="date", columns="ticker", values="flag", aggfunc="first")
        .reindex(wide.index)
        .reindex(columns=wide.columns)
        .fillna(False)
        .astype(bool)
    )
    PIT_RETURNS_WIDE = wide
    PIT_MASK = mask
    # Coverage is reported two ways:
    # 1) selected coverage: member-days after restricting to downloaded/usable columns;
    # 2) raw requested coverage: all PIT member-days requested before symbol/data filtering.
    selected_member_day_total = int(mask.sum().sum())
    selected_covered = int((mask & wide.notna()).sum().sum())
    coverage = selected_covered / max(selected_member_day_total, 1)

    raw_membership_mask = (
        MEMBERSHIP_LONG.assign(flag=True)
        .pivot_table(index="date", columns="ticker", values="flag", aggfunc="first")
        .reindex(wide.index)
        .fillna(False)
        .astype(bool)
    )
    raw_member_day_total = int(raw_membership_mask.sum().sum())
    raw_price_panel = _wide_returns_from_raw(raw, DATA_MODE).reindex(raw_membership_mask.index)
    raw_available = raw_membership_mask.reindex(columns=raw_price_panel.columns, fill_value=False) & raw_price_panel.notna()
    raw_covered = int(raw_available.sum().sum())
    raw_coverage = raw_covered / max(raw_member_day_total, 1)

    CONFIG["pit_member_day_coverage"] = round(float(coverage), 4)
    CONFIG["pit_member_day_coverage_basis"] = "selected_downloaded_symbols_after_usability_filter"
    CONFIG["pit_member_days_selected_total"] = int(selected_member_day_total)
    CONFIG["pit_member_days_selected_covered"] = int(selected_covered)
    CONFIG["pit_member_day_raw_coverage"] = round(float(raw_coverage), 4)
    CONFIG["pit_member_day_raw_coverage_basis"] = "all_pit_member_days_before_symbol_filter"
    CONFIG["pit_member_days_raw_total"] = int(raw_member_day_total)
    CONFIG["pit_member_days_raw_covered"] = int(raw_covered)
    CONFIG["pit_wide_symbols_with_data"] = int(len(keep_cols))
    USE_PIT_FACTORS = coverage >= 0.35
    CONFIG["use_pit_factors"] = bool(USE_PIT_FACTORS)
    sector_lookup = SECTOR_MAP if "SECTOR_MAP" in globals() else {}
    for col in wide.columns:
        sector = sector_lookup.get(col, "Unknown")
        PIT_SECTOR_COLUMNS.setdefault(sector, []).append(col)
    print(f"PIT wide panel: {wide.shape}, selected member-day data coverage {coverage:.1%}; raw PIT coverage {raw_coverage:.1%} -> USE_PIT_FACTORS={USE_PIT_FACTORS}")
    if coverage < 0.60:
        print("WARNING: material share of historical members lacks price data; factor tails remain partially survivorship-biased. Disclosed in manifest.")
else:
    CONFIG["use_pit_factors"] = False
    print("PIT factors disabled; factor frame will use the fixed loaded panel (disclosed).")


## 3. Regime Detection

Regime labels are used for classifier-free guidance. The model learns both:
- unconditional denoising;
- conditional denoising for bull/bear/crisis/recovery-like clusters.

We use a simple KMeans regime detector over rolling volatility, mean return, drawdown, and average correlation. You can replace this cell with an HMM or your own macro labels.
        


In [ ]:
#@title Regime labels: train-only HMM/KMeans with robust macro/VIX/rates fallback

def rolling_regime_features(returns: pd.DataFrame, lookback=60):
    port = returns.mean(axis=1)
    roll_mean = port.rolling(lookback, min_periods=max(20, lookback // 2)).mean()
    roll_vol = port.rolling(lookback, min_periods=max(20, lookback // 2)).std()

    eq = (1 + port.fillna(0)).cumprod()
    roll_peak = eq.rolling(lookback, min_periods=max(20, lookback // 2)).max()
    drawdown = eq / roll_peak - 1

    # Fast average-correlation proxy. Avoids the huge rolling corr matrix.
    n = returns.shape[1]
    asset_var = returns.rolling(lookback, min_periods=max(20, lookback // 2)).var().mean(axis=1)
    market_var = port.rolling(lookback, min_periods=max(20, lookback // 2)).var()
    avg_corr = ((n * market_var / asset_var.clip(lower=1e-12)) - 1) / max(n - 1, 1)
    avg_corr = avg_corr.clip(-1, 1)

    feats = pd.DataFrame({
        "mean": roll_mean,
        "vol": roll_vol,
        "drawdown": drawdown,
        "avg_corr": avg_corr,
    }, index=returns.index).replace([np.inf, -np.inf], np.nan)

    return feats


def compute_temporal_split_idx(index):
    """Return the first out-of-sample row. The row at split_idx is validation."""
    n = len(index)
    min_train = max(N_REGIMES * 20, WINDOW_SIZE + 252)

    if n <= min_train + WINDOW_SIZE:
        return max(1, int(n * TRAIN_FRACTION))

    if VALIDATION_MODE == "walk_forward_crisis":
        requested = int(np.searchsorted(index.values, np.datetime64(pd.Timestamp(WALK_FORWARD_TRAIN_END))))
        split_idx = max(requested, min_train)
        split_idx = min(split_idx, n - WINDOW_SIZE)
        return int(split_idx)

    return max(min_train, int(n * TRAIN_FRACTION))


def safe_yf_download_one(ticker, start, end):
    if not YFINANCE_AVAILABLE:
        return pd.DataFrame()

    for attempt in range(3):
        try:
            raw = yf.download(
                ticker,
                start=start,
                end=end,
                auto_adjust=False,
                progress=False,
                threads=False,
            )
            if raw is not None and len(raw):
                raw = raw.copy()
                raw.index = pd.to_datetime(raw.index)
                return raw
        except Exception as exc:
            print(f"yfinance failed for {ticker}, attempt {attempt + 1}/3:", str(exc)[:160])
            time.sleep(1.5 * (attempt + 1))

    print(f"Skipping optional macro ticker after retries: {ticker}")
    return pd.DataFrame()


def fetch_macro_context(index):
    if not USE_MACRO_CONDITIONING or not YFINANCE_AVAILABLE:
        return pd.DataFrame(index=index)

    start = (pd.Timestamp(index.min()) - pd.Timedelta(days=10)).strftime("%Y-%m-%d")
    end = (pd.Timestamp(index.max()) + pd.Timedelta(days=10)).strftime("%Y-%m-%d")
    out = pd.DataFrame(index=pd.DatetimeIndex(index))

    vix = safe_yf_download_one("^VIX", start, end)
    if len(vix) and "Close" in vix:
        s = vix["Close"].astype(float)
        s.name = "vix"
        out["vix"] = s.reindex(out.index).ffill()
        out["vix_change_5d"] = out["vix"].pct_change(5)

    tnx = safe_yf_download_one("^TNX", start, end)
    if len(tnx) and "Close" in tnx:
        s = (tnx["Close"].astype(float) / 100.0)
        s.name = "tnx_10y_pct"
        out["tnx_10y_pct"] = s.reindex(out.index).ffill()
        out["tnx_change_20d"] = out["tnx_10y_pct"].diff(20)

    spy = safe_yf_download_one("SPY", start, end)
    if len(spy) and "Volume" in spy:
        s = np.log1p(spy["Volume"].astype(float))
        s.name = "spy_volume_log"
        out["spy_volume_log"] = s.reindex(out.index).ffill()
        vol_std = out["spy_volume_log"].rolling(60, min_periods=20).std()
        out["spy_volume_z60"] = (
            out["spy_volume_log"] - out["spy_volume_log"].rolling(60, min_periods=20).mean()
        ) / vol_std.replace(0, np.nan)

    out = out.replace([np.inf, -np.inf], np.nan).ffill()
    return out


def fit_regime_model(X_train, method):
    method_used = method

    if method == "hmm" and HMM_AVAILABLE:
        try:
            hmm = GaussianHMM(
                n_components=N_REGIMES,
                covariance_type="diag",
                n_iter=500,
                tol=1e-4,
                random_state=SEED,
            )
            hmm.fit(X_train)
            labels_train = hmm.predict(X_train)
            return labels_train, "hmm", hmm
        except Exception as exc:
            print("HMM regime fit failed; falling back to KMeans:", str(exc)[:220])
            method_used = "kmeans"

    kmeans = KMeans(n_clusters=N_REGIMES, random_state=SEED, n_init=30)
    labels_train = kmeans.fit_predict(X_train)
    return labels_train, method_used, kmeans


def predict_regime_labels(model, X, method):
    if len(X) == 0:
        return np.array([], dtype=np.int64)
    if method == "hmm":
        return model.predict(X).astype(np.int64)
    return model.predict(X).astype(np.int64)


def regime_profile_from(features_frame, labels_series):
    joined = features_frame.join(labels_series.rename("regime"), how="inner")
    profile = joined.groupby("regime").agg(
        mean_return=("mean", "mean"),
        volatility=("vol", "mean"),
        drawdown=("drawdown", "mean"),
        avg_corr=("avg_corr", "mean"),
        count=("mean", "count"),
    )

    if "vix" in joined.columns:
        profile["vix"] = joined.groupby("regime")["vix"].mean()
    if "tnx_10y_pct" in joined.columns:
        profile["tnx_10y_pct"] = joined.groupby("regime")["tnx_10y_pct"].mean()
    if "spy_volume_z60" in joined.columns:
        profile["spy_volume_z60"] = joined.groupby("regime")["spy_volume_z60"].mean()

    profile["crisis_score"] = (
        profile["volatility"].rank(pct=True)
        + (-profile["drawdown"]).rank(pct=True)
        + profile["avg_corr"].rank(pct=True)
    )

    if "vix" in profile:
        profile["crisis_score"] += profile["vix"].rank(pct=True) * 0.50
    if "spy_volume_z60" in profile:
        profile["crisis_score"] += profile["spy_volume_z60"].rank(pct=True) * 0.25

    return profile.sort_values("crisis_score", ascending=False)


features_core = rolling_regime_features(returns_df)
macro_external = fetch_macro_context(features_core.index)

# Important: only core market features are mandatory.
# Optional macro columns must never erase the whole feature frame.
macro_features_raw = features_core.join(macro_external, how="left")
macro_features_raw = macro_features_raw.replace([np.inf, -np.inf], np.nan).sort_index()

core_cols = ["mean", "vol", "drawdown", "avg_corr"]
macro_cols = [
    "vix", "vix_change_5d",
    "tnx_10y_pct", "tnx_change_20d",
    "spy_volume_log", "spy_volume_z60",
]
available_macro_cols = [c for c in macro_cols if c in macro_features_raw.columns]

features = macro_features_raw[core_cols + available_macro_cols].copy()
features[available_macro_cols] = features[available_macro_cols].ffill()

# Drop rows only if core market features are missing.
features = features.dropna(subset=core_cols)

# Optional macro features get neutral train-safe fills.
for col in available_macro_cols:
    if features[col].isna().all():
        features = features.drop(columns=[col])
    else:
        features[col] = features[col].fillna(features[col].median())

regime_feature_cols = list(features.columns)

if len(features) == 0:
    raise RuntimeError(
        "Regime features are empty after robust fallback. "
        "Check returns_df shape and date index before this cell."
    )

regime_split_idx = compute_temporal_split_idx(features.index)
regime_train_features = features.iloc[:regime_split_idx]
regime_valid_features = features.iloc[regime_split_idx:]

min_train_rows = max(N_REGIMES * 20, WINDOW_SIZE + 120)
if len(regime_train_features) < min_train_rows:
    print(
        f"Warning: only {len(regime_train_features)} train rows for regime fitting; "
        f"using all available rows before validation split fallback."
    )
    regime_split_idx = max(1, min(len(features) - 1, int(len(features) * TRAIN_FRACTION)))
    regime_train_features = features.iloc[:regime_split_idx]
    regime_valid_features = features.iloc[regime_split_idx:]

if len(regime_train_features) < N_REGIMES * 10:
    raise RuntimeError(f"Still not enough train-only rows for regime fitting: {len(regime_train_features)}")

scaler = StandardScaler()
X_regime_train = scaler.fit_transform(regime_train_features)
X_regime_valid = (
    scaler.transform(regime_valid_features)
    if len(regime_valid_features)
    else np.empty((0, len(regime_feature_cols)))
)

labels_train, RESOLVED_REGIME_METHOD, regime_model = fit_regime_model(X_regime_train, REGIME_METHOD)
labels_valid = predict_regime_labels(regime_model, X_regime_valid, RESOLVED_REGIME_METHOD)
labels = np.concatenate([labels_train, labels_valid]).astype(np.int64)

regime_series = pd.Series(labels, index=features.index, name="regime")

CONFIG["regime_method_resolved"] = RESOLVED_REGIME_METHOD
CONFIG["regime_feature_cols"] = regime_feature_cols
CONFIG["regime_fit_scope"] = "train_only"
CONFIG["regime_train_start"] = str(regime_train_features.index.min().date())
CONFIG["regime_train_end"] = str(regime_train_features.index.max().date())
CONFIG["regime_valid_start"] = str(regime_valid_features.index.min().date()) if len(regime_valid_features) else None
CONFIG["regime_validation_labels"] = "predicted_by_train_fitted_model"
CONFIG["macro_optional_missing_tolerated"] = True

train_regime_series = regime_series.iloc[:regime_split_idx]
summary_train = regime_train_features.join(train_regime_series).groupby("regime").agg(["mean", "std", "count"])
print("Train-only regime feature summary:")
display(summary_train)

regime_profile_train = regime_profile_from(regime_train_features, train_regime_series)
print("Train-only regime crisis profile:")
display(regime_profile_train)

if AUTO_TARGET_CRISIS_REGIME:
    TARGET_REGIME = int(regime_profile_train["crisis_score"].idxmax())
    CONFIG["target_regime"] = TARGET_REGIME
    print("Auto-selected train-only crisis-like target regime:", TARGET_REGIME)

returns_aligned = returns_df.loc[regime_series.index]
macro_features_aligned = features.loc[returns_aligned.index].astype("float32")
MACRO_FEATURE_COLUMNS = list(macro_features_aligned.columns)
CONFIG["macro_feature_columns"] = MACRO_FEATURE_COLUMNS

print("Aligned returns:", returns_aligned.shape)
print("Macro conditioning features:", MACRO_FEATURE_COLUMNS)
print("Regime fit rows:", len(regime_train_features), "| Out-of-sample labeled rows:", len(regime_valid_features))


## 4. Dataset

Each training example is a rolling window of normalized multi-asset returns:

`x_0.shape = (window_size, n_assets)`

The DDPM forward process adds noise to that entire matrix; the score network learns to predict the noise.
        


In [ ]:
#@title Factor engine, datasets, and temporal split
class FactorWindowDataset(Dataset):
    def __init__(self, factor_frame: pd.DataFrame, regimes: pd.Series, macro_features: pd.DataFrame, window_size: int, factor_mu=None, factor_sigma=None, macro_mu=None, macro_sigma=None, label_start_date=None, asset_columns=None):
        self.factor_columns = list(factor_frame.columns)
        self.columns = list(asset_columns) if asset_columns is not None else []
        self.index = factor_frame.index
        x_raw = torch.tensor(factor_frame.values, dtype=torch.float32)
        self.factor_mu = x_raw.mean(dim=0) if factor_mu is None else factor_mu
        self.factor_sigma = x_raw.std(dim=0).clamp_min(1e-6) if factor_sigma is None else factor_sigma
        self.x = (x_raw - self.factor_mu) / self.factor_sigma
        self.factor_raw = x_raw
        macro = torch.tensor(macro_features.loc[factor_frame.index].values, dtype=torch.float32)
        self.macro_columns = list(macro_features.columns)
        self.macro_mu = macro.mean(dim=0) if macro_mu is None else macro_mu
        self.macro_sigma = macro.std(dim=0).clamp_min(1e-6) if macro_sigma is None else macro_sigma
        self.macro = torch.nan_to_num((macro - self.macro_mu) / self.macro_sigma, nan=0.0, posinf=0.0, neginf=0.0)
        self.regimes = torch.tensor(regimes.loc[factor_frame.index].values, dtype=torch.long)
        self.window_size = int(window_size)
        self.label_start_date = pd.Timestamp(label_start_date) if label_start_date is not None else None
        self.window_end_dates = pd.Index(self.index[self.window_size - 1:])

    def __len__(self):
        return max(0, len(self.x) - self.window_size + 1)

    def __getitem__(self, idx):
        end = idx + self.window_size
        return self.x[idx:end], self.macro[idx:end], self.regimes[end - 1]


class AssetWindowDataset(Dataset):
    def __init__(self, returns: pd.DataFrame, regimes: pd.Series, window_size: int):
        self.columns = list(returns.columns)
        self.index = returns.index
        self.x = torch.tensor(returns.values, dtype=torch.float32)
        self.regimes = torch.tensor(regimes.loc[returns.index].values, dtype=torch.long)
        self.window_size = int(window_size)
        self.window_end_dates = pd.Index(self.index[self.window_size - 1:])

    def __len__(self):
        return max(0, len(self.x) - self.window_size + 1)

    def __getitem__(self, idx):
        end = idx + self.window_size
        return self.x[idx:end], self.regimes[end - 1]


def sanitize_factor_name(x):
    return re.sub(r"[^A-Za-z0-9_]+", "_", str(x)).strip("_")[:40] or "unknown"


split_idx = compute_temporal_split_idx(returns_aligned.index)
valid_context_start_idx = max(0, split_idx - WINDOW_SIZE + 1)
valid_label_start = returns_aligned.index[split_idx]

train_returns = returns_aligned.iloc[:split_idx]
valid_returns = returns_aligned.iloc[valid_context_start_idx:]
train_regimes = regime_series.loc[train_returns.index]
valid_regimes = regime_series.loc[valid_returns.index]
train_macro = macro_features_aligned.loc[train_returns.index]
valid_macro = macro_features_aligned.loc[valid_returns.index]
asset_columns = list(train_returns.columns)

asset_mu_np = train_returns[asset_columns].mean(axis=0).values.astype(np.float32)
asset_sigma_np = train_returns[asset_columns].std(axis=0).replace(0, np.nan).fillna(1e-6).values.astype(np.float32)
asset_sigma_np = np.clip(asset_sigma_np, 1e-6, None)
X_train_std = ((train_returns[asset_columns].values.astype(np.float32) - asset_mu_np) / asset_sigma_np).astype(np.float32)

N_PCA_FACTORS = int(min(FACTOR_PCA_COMPONENTS, X_train_std.shape[0] - 2, X_train_std.shape[1] - 1))
pca = PCA(n_components=N_PCA_FACTORS, svd_solver="randomized", random_state=SEED)
pca.fit(X_train_std)

asset_sectors = pd.Series([SECTOR_MAP.get(c, "Unknown") if "SECTOR_MAP" in globals() else "Unknown" for c in asset_columns], index=asset_columns)
sector_names = sorted(asset_sectors.fillna("Unknown").unique().tolist())
sector_matrix = np.zeros((len(asset_columns), len(sector_names)), dtype=np.float32)
for j, sector in enumerate(sector_names):
    idx = np.where(asset_sectors.values == sector)[0]
    if len(idx):
        sector_matrix[idx, j] = 1.0 / len(idx)


def compute_factor_frame(returns_frame: pd.DataFrame) -> pd.DataFrame:
    X = returns_frame[asset_columns].values.astype(np.float32)
    if USE_PIT_FACTORS and PIT_RETURNS_WIDE is not None:
        wide = PIT_RETURNS_WIDE.reindex(returns_frame.index)
        mask = PIT_MASK.reindex(returns_frame.index).fillna(False)
        masked = wide.where(mask)
        fallback_market = pd.Series(X.mean(axis=1), index=returns_frame.index)
        market = masked.mean(axis=1).fillna(fallback_market).values.reshape(-1, 1).astype(np.float32)
        fallback_sector = X @ sector_matrix
        sector_series = []
        for j, sector_name in enumerate(sector_names):
            cols = [c for c in PIT_SECTOR_COLUMNS.get(sector_name, []) if c in masked.columns]
            if cols:
                s = masked[cols].mean(axis=1)
            else:
                s = pd.Series(np.nan, index=returns_frame.index)
            s = s.fillna(pd.Series(fallback_sector[:, j], index=returns_frame.index))
            sector_series.append(s.values.astype(np.float32))
        sector = np.stack(sector_series, axis=1)
    else:
        market = X.mean(axis=1, keepdims=True)
        sector = X @ sector_matrix
    X_std = ((X - asset_mu_np) / asset_sigma_np).astype(np.float32)
    pca_scores = pca.transform(X_std).astype(np.float32)
    values = np.concatenate([market, sector, pca_scores], axis=1).astype(np.float32)
    columns = ["market"] + [f"sector_{sanitize_factor_name(s)}" for s in sector_names] + [f"pca_{i+1:02d}" for i in range(N_PCA_FACTORS)]
    return pd.DataFrame(values, index=returns_frame.index, columns=columns)


FACTOR_RAW_ALIGNED = compute_factor_frame(returns_aligned)
train_factor = FACTOR_RAW_ALIGNED.loc[train_returns.index]
valid_factor = FACTOR_RAW_ALIGNED.loc[valid_returns.index]
FACTOR_COLUMNS = list(train_factor.columns)
N_FACTORS_ACTUAL = len(FACTOR_COLUMNS)

F_train = train_factor.values.astype(np.float64)
R_train = train_returns[asset_columns].values.astype(np.float64)
F_aug = np.concatenate([np.ones((len(F_train), 1)), F_train], axis=1)
ridge = np.eye(F_aug.shape[1]) * float(FACTOR_RIDGE_ALPHA)
ridge[0, 0] = 0.0
RECON_BETA = np.linalg.solve(F_aug.T @ F_aug + ridge, F_aug.T @ R_train).astype(np.float32)


def reconstruct_mean_np(factor_values):
    f = np.asarray(factor_values, dtype=np.float32)
    aug = np.concatenate([np.ones((*f.shape[:-1], 1), dtype=np.float32), f], axis=-1)
    return np.matmul(aug, RECON_BETA).astype(np.float32)


mean_recon_all = reconstruct_mean_np(FACTOR_RAW_ALIGNED.values.astype(np.float32))
RESIDUAL_RETURNS_ALIGNED = pd.DataFrame(
    returns_aligned[asset_columns].values.astype(np.float32) - mean_recon_all,
    index=returns_aligned.index,
    columns=asset_columns,
)
resid_train = RESIDUAL_RETURNS_ALIGNED.loc[train_returns.index].values.astype(np.float32)
resid_cov = np.cov(resid_train.T)
resid_diag = np.diag(np.diag(resid_cov))
resid_cov = (1 - RESIDUAL_COV_SHRINKAGE) * resid_cov + RESIDUAL_COV_SHRINKAGE * resid_diag + 1e-7 * np.eye(resid_cov.shape[0])

def safe_cholesky_np(mat):
    jitter = 1e-7
    eye = np.eye(mat.shape[0])
    for _ in range(8):
        try:
            return np.linalg.cholesky(mat + jitter * eye).astype(np.float32)
        except np.linalg.LinAlgError:
            jitter *= 10
    vals, vecs = np.linalg.eigh(mat)
    vals = np.clip(vals, 1e-8, None)
    return (vecs @ np.diag(np.sqrt(vals))).astype(np.float32)

RESIDUAL_COV_CHOL = safe_cholesky_np(resid_cov)

train_ds = FactorWindowDataset(train_factor, train_regimes, train_macro, WINDOW_SIZE, asset_columns=asset_columns)
valid_ds = FactorWindowDataset(
    valid_factor, valid_regimes, valid_macro, WINDOW_SIZE,
    train_ds.factor_mu, train_ds.factor_sigma, train_ds.macro_mu, train_ds.macro_sigma,
    label_start_date=valid_label_start, asset_columns=asset_columns,
)
asset_train_ds = AssetWindowDataset(train_returns, train_regimes, WINDOW_SIZE)
asset_valid_ds = AssetWindowDataset(valid_returns, valid_regimes, WINDOW_SIZE)

if len(valid_ds) and valid_ds.window_end_dates.min() < valid_label_start:
    raise RuntimeError("Validation windows include a pre-cutoff label date; split logic is leaking train labels into validation.")

N_ASSETS_ACTUAL = len(asset_columns)
N_MACRO_FEATURES = train_ds.macro.shape[-1]
CONFIG["n_assets_actual"] = N_ASSETS_ACTUAL
CONFIG["n_factors_actual"] = N_FACTORS_ACTUAL
CONFIG["n_macro_features"] = N_MACRO_FEATURES
CONFIG["factor_columns"] = FACTOR_COLUMNS
CONFIG["sector_factor_count"] = len(sector_names)
CONFIG["pca_factor_count"] = N_PCA_FACTORS
CONFIG["pca_explained_variance_ratio_sum"] = float(np.sum(pca.explained_variance_ratio_))
CONFIG["train_start"] = str(train_returns.index.min().date())
CONFIG["train_end"] = str(train_returns.index.max().date())
CONFIG["valid_context_start"] = str(valid_returns.index.min().date())
CONFIG["valid_label_start"] = str(valid_label_start.date())
CONFIG["valid_start"] = str(valid_label_start.date())
CONFIG["valid_end"] = str(valid_returns.index.max().date())
CONFIG["validation_protocol"] = "walk-forward labels start at valid_label_start; prior rows are context only"
CONFIG["model_family"] = MODEL_FAMILY

FACTOR_MU_T = train_ds.factor_mu.to(DEVICE)
FACTOR_SIGMA_T = train_ds.factor_sigma.to(DEVICE)
RECON_BETA_T = torch.tensor(RECON_BETA, dtype=torch.float32, device=DEVICE)
ASSET_CORR_PROJECTOR = torch.randn(N_ASSETS_ACTUAL, min(CORR_PROJECTION_DIM, N_ASSETS_ACTUAL), device=DEVICE) / math.sqrt(min(CORR_PROJECTION_DIM, N_ASSETS_ACTUAL))

train_window_regimes = np.array([int(train_ds[i][2]) for i in range(len(train_ds))])
regime_counts = pd.Series(train_window_regimes).value_counts().sort_index()
print("Train window regime counts:")
display(regime_counts.to_frame("windows"))

sample_weights = None
if USE_REGIME_BALANCED_SAMPLER:
    inv = {int(k): (len(train_window_regimes) / max(v, 1)) ** REGIME_SAMPLER_POWER for k, v in regime_counts.items()}
    sample_weights = np.array([inv[int(r)] for r in train_window_regimes], dtype=np.float64)
    if AUTO_TARGET_CRISIS_REGIME:
        sample_weights[train_window_regimes == int(TARGET_REGIME)] *= TARGET_REGIME_LOSS_BOOST
    sample_weights = sample_weights / sample_weights.mean()
    sampler = WeightedRandomSampler(
        weights=torch.as_tensor(sample_weights, dtype=torch.double),
        num_samples=len(sample_weights),
        replacement=True,
    )
    shuffle = False
else:
    sampler = None
    shuffle = True

REGIME_LOSS_WEIGHTS = torch.ones(N_REGIMES + 1, device=DEVICE)
for k, v in regime_counts.items():
    REGIME_LOSS_WEIGHTS[int(k)] = (float(regime_counts.max()) / max(float(v), 1.0)) ** REGIME_LOSS_POWER
if AUTO_TARGET_CRISIS_REGIME:
    REGIME_LOSS_WEIGHTS[int(TARGET_REGIME)] *= TARGET_REGIME_LOSS_BOOST
REGIME_LOSS_WEIGHTS[:N_REGIMES] = REGIME_LOSS_WEIGHTS[:N_REGIMES] / REGIME_LOSS_WEIGHTS[:N_REGIMES].mean().clamp_min(1e-6)
print("Regime loss weights:", {i: float(REGIME_LOSS_WEIGHTS[i].detach().cpu()) for i in range(N_REGIMES)})

pin_memory = DEVICE == "cuda"
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=shuffle, sampler=sampler,
    num_workers=NUM_WORKERS, pin_memory=pin_memory, drop_last=True,
)
valid_loader = DataLoader(
    valid_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=pin_memory, drop_last=False,
)

print("Train windows:", len(train_ds), "Valid windows:", len(valid_ds), "Assets:", N_ASSETS_ACTUAL, "Factors:", N_FACTORS_ACTUAL, "Macro features:", N_MACRO_FEATURES)
print("PCA explained variance:", round(CONFIG["pca_explained_variance_ratio_sum"], 4))
print("Train period:", CONFIG["train_start"], "to", CONFIG["train_end"])
print("Valid context:", CONFIG["valid_context_start"], "| Valid labels:", CONFIG["valid_label_start"], "to", CONFIG["valid_end"])


## 5. FHS factor engine (V9)

EWMA-devolatized factor returns, regime-conditioned block bootstrap of standardized innovations, vol state initialized from same-regime days. No neural network, no GPU, fully seeded.


In [ ]:
#@title V9.5 engine: regime-conditioned filtered historical simulation on factors
def ewma_vol_matrix(values, lam):
    arr = np.asarray(values, dtype=np.float64)
    vol = np.zeros_like(arr)
    vol[0] = np.nanstd(arr, axis=0) + 1e-8
    for t in range(1, len(arr)):
        vol[t] = np.sqrt(lam * vol[t - 1] ** 2 + (1 - lam) * arr[t - 1] ** 2)
    return np.maximum(vol, 1e-8)


class FhsFactorEngine:
    """Filtered historical simulation over the factor space.

    fit: devolatize factors with EWMA(lambda); store standardized innovations Z.
    simulate: sample innovation blocks from same-regime days, initialize the vol
    state from a same-regime day, and roll the EWMA recursion forward.
    """

    def __init__(self, factor_frame, regimes, lam, block_length, min_pool, seed):
        self.lam = float(lam)
        self.block = max(1, int(block_length))
        self.min_pool = int(min_pool)
        self.index = factor_frame.index
        self.F = factor_frame.values.astype(np.float64)
        self.vol = ewma_vol_matrix(self.F, self.lam)
        self.Z = self.F / self.vol
        self.reg = np.asarray(regimes.loc[factor_frame.index].values, dtype=int)
        self.rng = np.random.default_rng(seed)

    def _pool(self, regime):
        limit = len(self.Z) - self.block
        if regime is None:
            return np.arange(limit), "unconditional"
        idx = np.where(self.reg == int(regime))[0]
        idx = idx[idx < limit]
        if len(idx) >= self.min_pool:
            return idx, "regime_conditioned"
        return np.arange(limit), "all_train_fallback"

    def simulate(self, n_scenarios, horizon, regime, seed=None):
        rng = np.random.default_rng(seed) if seed is not None else self.rng
        pool, pool_note = self._pool(regime)
        n = int(n_scenarios)
        horizon = int(horizon)
        n_factors = self.Z.shape[1]
        n_blocks = int(np.ceil(horizon / self.block))
        starts = rng.choice(pool, size=(n, n_blocks), replace=True)
        z = np.empty((n, horizon, n_factors), dtype=np.float64)
        for b in range(n_blocks):
            for j in range(self.block):
                t = b * self.block + j
                if t >= horizon:
                    break
                z[:, t, :] = self.Z[starts[:, b] + j]
        init_idx = rng.choice(pool, size=n, replace=True)
        sig = self.vol[init_idx].copy()
        out = np.empty((n, horizon, n_factors), dtype=np.float64)
        for t in range(horizon):
            f = sig * z[:, t, :]
            out[:, t, :] = f
            sig = np.maximum(np.sqrt(self.lam * sig ** 2 + (1 - self.lam) * f ** 2), 1e-8)
        return out.astype(np.float32), pool_note


FHS_ENGINE = FhsFactorEngine(
    train_factor, train_regimes,
    lam=FHS_EWMA_LAMBDA, block_length=FHS_BLOCK_LENGTH,
    min_pool=FHS_MIN_REGIME_POOL, seed=SEED,
)
_pool_idx, _pool_note = FHS_ENGINE._pool(TARGET_REGIME)
CONFIG["fhs_target_regime_pool_days"] = int(len(_pool_idx))
CONFIG["fhs_target_regime_pool_note"] = _pool_note
print("FHS engine fitted:", FHS_ENGINE.Z.shape, "| target-regime pool:", len(_pool_idx), _pool_note)


## 7. Sample synthetic stress scenarios (FHS)


In [ ]:
#@title FHS factor sampler, tail anchoring, catastrophe sleeve, Pareto Cholesky sweep, and gate books (V9.7)
def as_numpy_local(x):
    return x.detach().cpu().numpy() if torch.is_tensor(x) else np.asarray(x)

def denormalize_factor_windows(x_norm):
    arr = as_numpy_local(x_norm).astype(np.float32)
    return arr * as_numpy_local(train_ds.factor_sigma)[None, None, :] + as_numpy_local(train_ds.factor_mu)[None, None, :]

def dataset_factor_windows(ds, max_samples=None, regime_filter=None, seed=SEED):
    idxs = list(range(len(ds)))
    if regime_filter is not None:
        idxs = [i for i in idxs if int(ds[i][2]) == int(regime_filter)]
    if not idxs:
        return torch.empty(0, WINDOW_SIZE, N_FACTORS_ACTUAL)
    rng = np.random.default_rng(seed)
    if max_samples is not None and len(idxs) > max_samples:
        idxs = rng.choice(idxs, size=max_samples, replace=False).tolist()
    return torch.stack([ds[i][0] for i in idxs])

def dataset_asset_windows(ds, max_samples=None, regime_filter=None, seed=SEED):
    idxs = list(range(len(ds)))
    if regime_filter is not None:
        idxs = [i for i in idxs if int(ds[i][1]) == int(regime_filter)]
    if not idxs:
        return torch.empty(0, WINDOW_SIZE, N_ASSETS_ACTUAL)
    rng = np.random.default_rng(seed)
    if max_samples is not None and len(idxs) > max_samples:
        idxs = rng.choice(idxs, size=max_samples, replace=False).tolist()
    return torch.stack([ds[i][0] for i in idxs])

def macro_condition_window(ds, target_regime, seed=SEED):
    # V8: sample a real macro trajectory instead of averaging dynamic windows into a static template.
    idxs = [i for i in range(len(ds)) if int(ds[i][2]) == int(target_regime)]
    if len(idxs) < 16:
        idxs = list(range(len(ds)))
    rng = np.random.default_rng(seed)
    chosen = int(rng.choice(idxs))
    CONFIG["macro_condition_source"] = "sampled_train_macro_window"
    CONFIG["macro_condition_window_end"] = str(ds.window_end_dates[chosen]) if hasattr(ds, "window_end_dates") else None
    return ds[chosen][1]

def quick_portfolio_summary(samples, weights=None):
    arr = as_numpy_local(samples)
    if len(arr) == 0:
        return {
            "mean_terminal": np.nan, "median_terminal": np.nan, "vol_daily": np.nan,
            "var5": np.nan, "cvar5": np.nan, "prob_loss": np.nan, "prob_drawdown_10pct": np.nan,
        }
    if weights is None:
        weights = np.ones(arr.shape[-1]) / arr.shape[-1]
    daily = np.clip(arr @ weights, -0.95, 1.50)
    terminal = np.prod(1 + daily, axis=1) - 1
    wealth = np.cumprod(1 + daily, axis=1)
    drawdown = wealth / np.maximum.accumulate(wealth, axis=1) - 1
    var5 = np.quantile(terminal, 0.05)
    return {
        "mean_terminal": float(np.mean(terminal)),
        "median_terminal": float(np.median(terminal)),
        "vol_daily": float(np.std(daily)),
        "var5": float(var5),
        "cvar5": float(terminal[terminal <= var5].mean()),
        "prob_loss": float((terminal < 0).mean()),
        "prob_drawdown_10pct": float((drawdown.min(axis=1) <= -0.10).mean()),
    }

def parse_float_list(text):
    return np.array([float(x.strip()) for x in str(text).split(",") if x.strip()], dtype=np.float32)

def stratified_stress_multipliers(n, weights, multipliers, seed=SEED):
    rng = np.random.default_rng(seed)
    weights = np.asarray(weights, dtype=np.float64)
    multipliers = np.asarray(multipliers, dtype=np.float32)
    weights = weights / weights.sum()
    if not STRESS_STRATIFIED_SAMPLING:
        return rng.choice(multipliers, size=int(n), replace=True, p=weights).astype(np.float32)
    raw_counts = weights * int(n)
    counts = np.floor(raw_counts).astype(int)
    remainder = int(n) - int(counts.sum())
    if remainder > 0:
        order = np.argsort(-(raw_counts - counts))
        counts[order[:remainder]] += 1
    elif remainder < 0:
        order = np.argsort(raw_counts - counts)
        for idx in order[:abs(remainder)]:
            if counts[idx] > 0:
                counts[idx] -= 1
    book = np.repeat(multipliers, counts)
    if len(book) < int(n):
        book = np.concatenate([book, rng.choice(multipliers, size=int(n) - len(book), replace=True, p=weights)])
    elif len(book) > int(n):
        book = book[: int(n)]
    rng.shuffle(book)
    return book.astype(np.float32)

def stress_ladder(factor_raw, seed=SEED):
    f = np.asarray(factor_raw, dtype=np.float32).copy()
    if not STRESS_SCENARIO_MODE:
        return f, np.ones(f.shape[0], dtype=np.float32)
    weights = parse_float_list(STRESS_MIX_WEIGHTS)
    multipliers = parse_float_list(STRESS_MIX_MULTIPLIERS)
    weights = weights / weights.sum()
    m = stratified_stress_multipliers(f.shape[0], weights, multipliers, seed=seed)
    center = train_factor.median(axis=0).values.astype(np.float32)
    scale = np.sqrt(m)[:, None, None]
    f[:, :, 1:] = center[None, None, 1:] + (f[:, :, 1:] - center[None, None, 1:]) * scale
    market_center = center[0]
    downside = f[:, :, 0] < market_center
    down_scale = (m[:, None] * DOWNSIDE_ASYMMETRY).astype(np.float32)
    up_scale = (0.85 + 0.15 / np.maximum(m[:, None], 1e-6)).astype(np.float32)
    f[:, :, 0] = np.where(
        downside,
        market_center + (f[:, :, 0] - market_center) * down_scale,
        market_center + (f[:, :, 0] - market_center) * up_scale,
    )

    # V9.5: PIT crises exposed a centering failure. Add a train-only market-factor
    # location shift that grows with the stress multiplier. This is not a deterministic
    # full-window floor; it moves the adverse distribution using only train factor tails.
    train_market = train_factor.iloc[:, 0].astype(np.float32).values
    max_extra = max(float(np.max(multipliers)) - float(STRESS_LOCATION_SHIFT_MIN_MULTIPLIER), 1e-6)
    severity = np.clip((m - float(STRESS_LOCATION_SHIFT_MIN_MULTIPLIER)) / max_extra, 0.0, 1.0).astype(np.float32)
    tail_anchor = float(np.quantile(train_market, STRESS_LOCATION_SHIFT_Q))
    raw_shift = (tail_anchor - market_center) * float(STRESS_LOCATION_SHIFT_ALPHA) * severity
    raw_shift = np.clip(raw_shift, -float(STRESS_LOCATION_SHIFT_MAX_DAILY_ABS), 0.0).astype(np.float32)
    f[:, :, 0] = f[:, :, 0] + raw_shift[:, None]
    CONFIG["stress_location_shift_applied"] = True
    CONFIG["stress_location_shift_tail_anchor"] = tail_anchor
    CONFIG["stress_location_shift_mean_daily"] = float(np.mean(raw_shift))
    CONFIG["stress_location_shift_min_daily"] = float(np.min(raw_shift))

    # V8/V9 crisis capsule: sparse location/scale stress, never a full-window deterministic floor.
    severe = m >= float(np.max(multipliers))
    if severe.any():
        rng = np.random.default_rng(seed + 999)
        q_shift = float(np.quantile(train_market, SEVERE_MARKET_LOCATION_SHIFT_Q))
        for row in np.where(severe)[0]:
            n_days = int(rng.integers(SEVERE_SHOCK_MIN_DAYS, SEVERE_SHOCK_MAX_DAYS + 1))
            n_days = int(np.clip(n_days, 1, f.shape[1]))
            shock_days = rng.choice(f.shape[1], size=n_days, replace=False)
            local = f[row, shock_days, 0]
            centered = local - np.median(train_market)
            f[row, shock_days, 0] = q_shift + centered * SEVERE_MARKET_SCALE_BOOST
    CONFIG["stress_full_window_floor_applied"] = False
    CONFIG["stress_shock_mode_applied"] = STRESS_SHOCK_MODE
    return f, m

def train_only_catastrophe_daily_shock():
    if "train_returns" not in globals() or len(train_returns) < 64:
        return -float(STRESS_CATASTROPHE_MIN_DAILY_ABS)
    ew = np.clip(train_returns.values.astype(np.float64) @ (np.ones(train_returns.shape[1]) / train_returns.shape[1]), -0.95, 1.50)
    q = float(np.quantile(ew, STRESS_CATASTROPHE_DAILY_Q))
    shock = -abs(q) * float(STRESS_CATASTROPHE_SHOCK_SCALE)
    shock = float(np.clip(shock, -float(STRESS_CATASTROPHE_MAX_DAILY_ABS), -float(STRESS_CATASTROPHE_MIN_DAILY_ABS)))
    CONFIG["stress_catastrophe_train_daily_q"] = q
    CONFIG["stress_catastrophe_daily_shock"] = shock
    return shock

def catastrophe_row_mask(n, multipliers=None, seed=SEED):
    n = int(n)
    mask = np.zeros(n, dtype=bool)
    if n <= 0 or not STRESS_CATASTROPHE_SLEEVE:
        return mask
    count = max(1, int(round(n * float(STRESS_CATASTROPHE_WEIGHT))))
    rng = np.random.default_rng(seed + 4242)
    if multipliers is not None and len(multipliers) == n:
        m = np.asarray(multipliers, dtype=np.float32)
        order = np.argsort(-m)
        # deterministic top-multiplier sleeve, shuffled only inside ties by tiny noise
        jitter = rng.normal(0, 1e-6, size=n)
        order = np.lexsort((jitter, -m))
        chosen = order[:count]
    else:
        chosen = rng.choice(n, size=count, replace=False)
    mask[chosen] = True
    return mask

def apply_catastrophe_sleeve_to_returns(returns, multipliers=None, seed=SEED):
    arr = np.asarray(returns, dtype=np.float32).copy()
    if not STRESS_CATASTROPHE_SLEEVE or len(arr) == 0:
        CONFIG["stress_catastrophe_sleeve_applied"] = False
        return arr
    mask = catastrophe_row_mask(len(arr), multipliers=multipliers, seed=seed)
    if not mask.any():
        CONFIG["stress_catastrophe_sleeve_applied"] = False
        return arr
    rng = np.random.default_rng(seed + 5252)
    shock = train_only_catastrophe_daily_shock()
    counts = []
    for row in np.where(mask)[0]:
        n_days = int(rng.integers(STRESS_CATASTROPHE_MIN_DAYS, STRESS_CATASTROPHE_MAX_DAYS + 1))
        n_days = int(np.clip(n_days, 1, arr.shape[1]))
        days = rng.choice(arr.shape[1], size=n_days, replace=False)
        arr[row, days, :] = arr[row, days, :] + shock
        counts.append(n_days)
    CONFIG["stress_catastrophe_sleeve_applied"] = True
    CONFIG["stress_catastrophe_sleeve_count"] = int(mask.sum())
    CONFIG["stress_catastrophe_sleeve_weight_realized"] = float(mask.mean())
    CONFIG["stress_catastrophe_days_mean"] = float(np.mean(counts)) if counts else 0.0
    CONFIG["stress_catastrophe_is_full_window_floor"] = False
    return np.clip(arr, -0.80, 1.50).astype(np.float32)

def residual_window_bank(regime_filter=None):
    residual_train = RESIDUAL_RETURNS_ALIGNED.loc[train_returns.index].values.astype(np.float32)
    reg = train_regimes.values.astype(int)
    windows = []
    for i in range(0, len(residual_train) - WINDOW_SIZE + 1):
        if regime_filter is None or int(reg[i + WINDOW_SIZE - 1]) == int(regime_filter):
            windows.append(residual_train[i:i + WINDOW_SIZE])
    if not windows:
        return np.empty((0, WINDOW_SIZE, N_ASSETS_ACTUAL), dtype=np.float32)
    return np.stack(windows).astype(np.float32)

RESIDUAL_WINDOW_BANK_TARGET = residual_window_bank(TARGET_REGIME)
RESIDUAL_WINDOW_BANK_ALL = residual_window_bank(None)

def reconstruct_asset_windows_from_factors(factor_raw, add_residual=True, stress=False, seed=SEED, residual_bank=None):
    factors = np.asarray(factor_raw, dtype=np.float32)
    if stress:
        factors, multipliers = stress_ladder(factors, seed=seed)
    else:
        multipliers = np.ones(factors.shape[0], dtype=np.float32)
    base = reconstruct_mean_np(factors)
    if add_residual:
        if residual_bank is not None:
            bank = residual_bank
        else:
            bank = RESIDUAL_WINDOW_BANK_TARGET if len(RESIDUAL_WINDOW_BANK_TARGET) >= 32 else RESIDUAL_WINDOW_BANK_ALL
        rng = np.random.default_rng(seed + 17)
        idx = rng.choice(len(bank), size=factors.shape[0], replace=True)
        resid = bank[idx] * float(RESIDUAL_BOOTSTRAP_SCALE) * np.sqrt(multipliers)[:, None, None]
        base = base + resid.astype(np.float32)
    if stress:
        base = apply_catastrophe_sleeve_to_returns(base, multipliers=multipliers, seed=seed)
    return torch.tensor(np.clip(base, -0.80, 1.50), dtype=torch.float32), multipliers

def sym_matrix_power(mat, power, eps=1e-7):
    mat = np.asarray(mat, dtype=np.float64)
    mat = 0.5 * (mat + mat.T)
    vals, vecs = np.linalg.eigh(mat)
    vals = np.clip(vals, eps, None)
    return (vecs * (vals ** power)) @ vecs.T

def shrink_cov_np(x, shrinkage=0.10):
    x = np.asarray(x, dtype=np.float64)
    if x.ndim != 2 or len(x) < 2:
        return np.eye(x.shape[-1] if x.ndim == 2 else N_FACTORS_ACTUAL, dtype=np.float64)
    cov = np.cov(x, rowvar=False)
    diag = np.diag(np.diag(cov))
    return (1.0 - shrinkage) * cov + shrinkage * diag + 1e-7 * np.eye(cov.shape[0])

def calibrate_factor_windows(synthetic, target, alpha=FACTOR_CALIBRATION_ALPHA, shrinkage=FACTOR_CALIBRATION_SHRINKAGE):
    syn = np.asarray(synthetic, dtype=np.float32)
    tgt = np.asarray(target, dtype=np.float32)
    if len(syn) == 0 or len(tgt) < 8:
        return syn
    shape = syn.shape
    s = syn.reshape(-1, shape[-1]).astype(np.float64)
    t = tgt.reshape(-1, tgt.shape[-1]).astype(np.float64)
    mu_s = s.mean(axis=0, keepdims=True)
    mu_t = t.mean(axis=0, keepdims=True)
    cs = shrink_cov_np(s - mu_s, shrinkage=shrinkage)
    ct = shrink_cov_np(t - mu_t, shrinkage=shrinkage)
    whiten = sym_matrix_power(cs, -0.5)
    recolor = sym_matrix_power(ct, 0.5)
    matched = (s - mu_s) @ whiten @ recolor + mu_t
    blended = (1.0 - float(alpha)) * s + float(alpha) * matched
    lo = np.quantile(t, 0.001, axis=0, keepdims=True)
    hi = np.quantile(t, 0.999, axis=0, keepdims=True)
    pad = 0.50 * np.maximum(hi - lo, 1e-6)
    blended = np.clip(blended, lo - pad, hi + pad)
    return blended.reshape(shape).astype(np.float32)

def apply_factor_tail_location_anchor(synthetic, target, alpha=FACTOR_TAIL_LOCATION_ALPHA):
    """Train-only lower-tail correction for the market factor after PIT activation."""
    syn = np.asarray(synthetic, dtype=np.float32).copy()
    tgt = np.asarray(target, dtype=np.float32)
    if len(syn) == 0 or len(tgt) < 32 or float(alpha) <= 0:
        CONFIG["factor_tail_location_anchor_applied"] = False
        return syn
    syn_terminal = np.sum(syn[:, :, 0], axis=1)
    tgt_terminal = np.sum(tgt[:, :, 0], axis=1)
    syn_q = float(np.quantile(syn_terminal, FACTOR_TAIL_ANCHOR_QUANTILE))
    tgt_q = float(np.quantile(tgt_terminal, FACTOR_TAIL_ANCHOR_QUANTILE))
    delta = tgt_q - syn_q
    if not np.isfinite(delta) or delta >= 0:
        CONFIG["factor_tail_location_anchor_applied"] = False
        CONFIG["factor_tail_location_delta"] = float(delta) if np.isfinite(delta) else None
        return syn
    median = float(np.median(syn_terminal))
    denom = max(median - syn_q, 1e-6)
    severity = np.clip((median - syn_terminal) / denom, 0.0, 1.0).astype(np.float32)
    daily_shift = np.clip(float(alpha) * delta / syn.shape[1], -float(FACTOR_TAIL_ANCHOR_MAX_DAILY_SHIFT), 0.0)
    syn[:, :, 0] = syn[:, :, 0] + daily_shift * severity[:, None]
    CONFIG["factor_tail_location_anchor_applied"] = True
    CONFIG["factor_tail_location_delta"] = float(delta)
    CONFIG["factor_tail_location_daily_shift"] = float(daily_shift)
    CONFIG["factor_tail_location_reference_q"] = float(tgt_q)
    CONFIG["factor_tail_location_synthetic_q_before"] = float(syn_q)
    return syn

def calibrate_asset_correlation_windows(samples, target, alpha=CHOLESKY_CALIBRATION_ALPHA, shrinkage=CHOLESKY_SHRINKAGE):
    """Train-only correlation transport. Preserves synthetic marginals more than full covariance matching."""
    syn = to_numpy(samples).astype(np.float32) if "to_numpy" in globals() else as_numpy_local(samples).astype(np.float32)
    tgt = to_numpy(target).astype(np.float32) if "to_numpy" in globals() else as_numpy_local(target).astype(np.float32)
    if not APPLY_CHOLESKY_CALIBRATION or len(syn) == 0 or len(tgt) < 16:
        return samples

    shape = syn.shape
    s = syn.reshape(-1, shape[-1]).astype(np.float64)
    t = tgt.reshape(-1, tgt.shape[-1]).astype(np.float64)
    mu_s = s.mean(axis=0, keepdims=True)
    sd_s = s.std(axis=0, keepdims=True) + 1e-7
    mu_t = t.mean(axis=0, keepdims=True)
    sd_t = t.std(axis=0, keepdims=True) + 1e-7

    zs = (s - mu_s) / sd_s
    zt = (t - mu_t) / sd_t
    corr_s = shrink_cov_np(zs, shrinkage=shrinkage)
    corr_t = shrink_cov_np(zt, shrinkage=shrinkage)
    matched_z = zs @ sym_matrix_power(corr_s, -0.5) @ sym_matrix_power(corr_t, 0.5)

    # Keep the DDPM/base marginal scale and tail level; transport mostly the cross-sectional dependence.
    transported = matched_z * sd_s + mu_s
    blended = (1.0 - float(alpha)) * s + float(alpha) * transported

    lo = np.quantile(s, 0.0005, axis=0, keepdims=True)
    hi = np.quantile(s, 0.9995, axis=0, keepdims=True)
    pad = 0.35 * np.maximum(hi - lo, 1e-6)
    blended = np.clip(blended, lo - pad, hi + pad).reshape(shape).astype(np.float32)
    return torch.tensor(blended, dtype=torch.float32)


target_factor_norm = dataset_factor_windows(train_ds, max_samples=CALIBRATION_TARGET_WINDOWS, regime_filter=TARGET_REGIME)
target_factor_source = "train_target_regime_factor_windows"
if len(target_factor_norm) < 64:
    target_factor_norm = dataset_factor_windows(train_ds, max_samples=CALIBRATION_TARGET_WINDOWS)
    target_factor_source = "train_all_regimes_factor_windows_fallback"
if len(target_factor_norm) < 64:
    raise RuntimeError("Not enough train factor windows for factor calibration target.")
target_factor_raw = denormalize_factor_windows(target_factor_norm)
CONFIG["factor_calibration_applied"] = True
CONFIG["factor_calibration_alpha"] = float(FACTOR_CALIBRATION_ALPHA)
CONFIG["factor_calibration_shrinkage"] = float(FACTOR_CALIBRATION_SHRINKAGE)
CONFIG["factor_calibration_target_source"] = target_factor_source
CONFIG["factor_calibration_uses_validation"] = False
print("Train factor calibration target:", target_factor_source, tuple(target_factor_raw.shape))

target_real_returns = dataset_asset_windows(asset_train_ds, max_samples=CALIBRATION_TARGET_WINDOWS, regime_filter=TARGET_REGIME)
target_source = "train_target_regime_asset_windows"
if len(target_real_returns) < 64:
    target_real_returns = dataset_asset_windows(asset_train_ds, max_samples=CALIBRATION_TARGET_WINDOWS)
    target_source = "train_all_regimes_asset_windows_fallback"
if len(target_real_returns) < 64:
    raise RuntimeError("Not enough train asset windows for calibration target.")

target_summary = quick_portfolio_summary(target_real_returns)
CONFIG["calibration_target_source"] = target_source
CONFIG["guidance_uses_validation"] = False
CONFIG["selected_guidance_scale"] = 1.0
print("Train empirical calibration target:", target_source)
print(json.dumps(target_summary, indent=2))

# ---- V9 generation: FHS factor engine replaces the DDPM sampler ----
synthetic_factor_raw_uncalibrated, fhs_pool_note = FHS_ENGINE.simulate(
    N_SCENARIOS, WINDOW_SIZE, TARGET_REGIME, seed=SEED + 404,
)
CONFIG["fhs_innovation_pool_used"] = fhs_pool_note
synthetic_factor_raw = calibrate_factor_windows(synthetic_factor_raw_uncalibrated, target_factor_raw)
synthetic_factor_raw = apply_factor_tail_location_anchor(synthetic_factor_raw, target_factor_raw)

synthetic_returns_base, base_stress_multipliers = reconstruct_asset_windows_from_factors(
    synthetic_factor_raw, add_residual=True, stress=False, seed=SEED + 101,
)
synthetic_returns, stress_multipliers = reconstruct_asset_windows_from_factors(
    synthetic_factor_raw, add_residual=True, stress=STRESS_SCENARIO_MODE, seed=SEED + 202,
)
synthetic_returns_base_uncalibrated, _ = reconstruct_asset_windows_from_factors(
    synthetic_factor_raw_uncalibrated, add_residual=True, stress=False, seed=SEED + 101,
)
synthetic_returns_uncalibrated, _ = reconstruct_asset_windows_from_factors(
    synthetic_factor_raw_uncalibrated, add_residual=True, stress=STRESS_SCENARIO_MODE, seed=SEED + 202,
)

# ---- V9.5 train-only Cholesky alpha sweep -----------------------------------
# Selection criterion lives entirely on TRAIN data: candidate correlation error vs the
# train target windows, benchmarked against a Gaussian sample fitted on those same
# train windows. Validation data never touches this choice.
def corr_mae_windows(a, b):
    A = as_numpy_local(a).astype(np.float64).reshape(-1, N_ASSETS_ACTUAL)
    B = as_numpy_local(b).astype(np.float64).reshape(-1, N_ASSETS_ACTUAL)
    ca = np.nan_to_num(np.corrcoef(A.T), nan=0.0)
    cb = np.nan_to_num(np.corrcoef(B.T), nan=0.0)
    m = ~np.eye(ca.shape[0], dtype=bool)
    return float(np.abs(ca[m] - cb[m]).mean())

def quick_projected_mmd_windows(a, b, seed=SEED, proj_dim=8, max_n=400):
    A = as_numpy_local(a).astype(np.float64)
    B = as_numpy_local(b).astype(np.float64)
    n = min(len(A), len(B), int(max_n))
    if n < 8:
        return np.inf
    rng = np.random.default_rng(seed)
    ia = rng.choice(len(A), size=n, replace=False) if len(A) > n else np.arange(n)
    ib = rng.choice(len(B), size=n, replace=False) if len(B) > n else np.arange(n)
    A = A[ia].reshape(n, -1)
    B = B[ib].reshape(n, -1)
    D = A.shape[1]
    P = rng.normal(size=(D, min(int(proj_dim), D))) / np.sqrt(max(1, min(int(proj_dim), D)))
    A = A @ P
    B = B @ P
    stacked = np.concatenate([A, B], axis=0)
    scale = np.median(np.abs(stacked - np.median(stacked, axis=0))) + 1e-6
    gamma = 1.0 / (2.0 * scale * scale * A.shape[1])
    def kmean(X, Y):
        xx = np.sum(X * X, axis=1, keepdims=True)
        yy = np.sum(Y * Y, axis=1, keepdims=True).T
        dist = np.maximum(xx + yy - 2 * X @ Y.T, 0.0)
        return float(np.exp(-gamma * dist).mean())
    return float(kmean(A, A) + kmean(B, B) - 2.0 * kmean(A, B))

def cvar_abs_gap_windows(a, b):
    qa = quick_portfolio_summary(a)["cvar5"]
    qb = quick_portfolio_summary(b)["cvar5"]
    if not np.isfinite(qa) or not np.isfinite(qb):
        return np.inf
    return float(abs(qa - qb))

def apply_daily_var5_safety_margin(samples, reference, label="book"):
    """Train-only tail reserve for daily VaR gates; leaves non-tail days untouched.

    V9.7 removes the validation-selected 1.60 multiplier. The target q05 is
    derived from train tail shape only: q05 minus a fixed share of the train
    q01-q05 tail gap, clipped by predeclared train-only multiplier bounds.
    """
    arr = as_numpy_local(samples).astype(np.float32).copy()
    ref = as_numpy_local(reference).astype(np.float32)
    if not APPLY_DAILY_VAR5_SAFETY_MARGIN or len(arr) == 0 or len(ref) < 16:
        CONFIG[f"daily_var5_safety_{label}_applied"] = False
        return samples
    w = np.ones(arr.shape[-1], dtype=np.float64) / arr.shape[-1]
    syn_daily = np.clip(arr @ w, -0.95, 1.50)
    ref_daily = np.clip(ref @ w, -0.95, 1.50)
    syn_q05 = float(np.quantile(syn_daily.reshape(-1), 0.05))
    ref_q05 = float(np.quantile(ref_daily.reshape(-1), 0.05))
    ref_tail_q = float(np.quantile(ref_daily.reshape(-1), float(DAILY_VAR5_SAFETY_TAIL_Q)))

    if ref_q05 < 0 and ref_tail_q < ref_q05:
        train_tail_gap = float(ref_q05 - ref_tail_q)
        target_q05_raw = ref_q05 - float(DAILY_VAR5_SAFETY_STEEPNESS_SHARE) * train_tail_gap
        lower = ref_q05 * float(DAILY_VAR5_SAFETY_MAX_MULTIPLIER)
        upper = ref_q05 * float(DAILY_VAR5_SAFETY_MIN_MULTIPLIER)
        target_q05 = float(np.clip(target_q05_raw, lower, upper))
        implied_multiplier = float(target_q05 / ref_q05)
    else:
        train_tail_gap = 0.0
        target_q05_raw = ref_q05
        target_q05 = ref_q05
        implied_multiplier = 1.0

    shift = float(np.clip(target_q05 - syn_q05, -float(DAILY_VAR5_SAFETY_MAX_SHIFT), 0.0))
    if shift >= 0:
        CONFIG[f"daily_var5_safety_{label}_applied"] = False
        CONFIG[f"daily_var5_safety_{label}_shift"] = 0.0
        CONFIG[f"daily_var5_safety_{label}_method"] = DAILY_VAR5_SAFETY_METHOD
        CONFIG[f"daily_var5_safety_{label}_uses_validation"] = False
        return torch.tensor(arr, dtype=torch.float32) if torch.is_tensor(samples) else arr
    mask = syn_daily <= syn_q05
    arr[mask, :] = arr[mask, :] + shift
    CONFIG[f"daily_var5_safety_{label}_applied"] = True
    CONFIG[f"daily_var5_safety_{label}_shift"] = shift
    CONFIG[f"daily_var5_safety_{label}_synthetic_q05_before"] = syn_q05
    CONFIG[f"daily_var5_safety_{label}_train_ref_q05"] = ref_q05
    CONFIG[f"daily_var5_safety_{label}_train_ref_tail_q"] = ref_tail_q
    CONFIG[f"daily_var5_safety_{label}_train_tail_gap"] = train_tail_gap
    CONFIG[f"daily_var5_safety_{label}_target_q05_raw"] = target_q05_raw
    CONFIG[f"daily_var5_safety_{label}_target_q05"] = target_q05
    CONFIG[f"daily_var5_safety_{label}_implied_multiplier"] = implied_multiplier
    CONFIG[f"daily_var5_safety_{label}_method"] = DAILY_VAR5_SAFETY_METHOD
    CONFIG[f"daily_var5_safety_{label}_uses_validation"] = False
    CONFIG[f"daily_var5_safety_{label}_tail_day_share"] = float(mask.mean())
    out = np.clip(arr, -0.80, 1.50).astype(np.float32)
    return torch.tensor(out, dtype=torch.float32) if torch.is_tensor(samples) else out

CHOLESKY_SWEEP_ROWS = []
selected_cholesky_alpha = float(CHOLESKY_CALIBRATION_ALPHA)
if APPLY_CHOLESKY_CALIBRATION:
    _tgt = as_numpy_local(target_real_returns).astype(np.float64)
    _flat = _tgt.reshape(-1, _tgt.shape[-1])
    _mu = _flat.mean(axis=0)
    _cov = np.cov(_flat.T)
    _cov = 0.95 * _cov + 0.05 * np.diag(np.diag(_cov)) + 1e-6 * np.eye(_cov.shape[0])
    _L = np.linalg.cholesky(_cov)
    _rngs = np.random.default_rng(SEED + 909)
    _gauss = _rngs.normal(size=(min(600, len(_tgt)), WINDOW_SIZE, _tgt.shape[-1])) @ _L.T + _mu
    benchmark_mae = corr_mae_windows(_gauss, target_real_returns)
    _sub = synthetic_returns_base_uncalibrated[:CHOLESKY_SWEEP_SUBSAMPLE]
    for _alpha in [float(x) for x in str(CHOLESKY_ALPHA_SWEEP_VALUES).split(",") if x.strip()]:
        _cand = calibrate_asset_correlation_windows(_sub, target_real_returns, alpha=_alpha, shrinkage=CHOLESKY_SHRINKAGE)
        _mae = corr_mae_windows(_cand, target_real_returns)
        _mmd = quick_projected_mmd_windows(_cand, target_real_returns, seed=SEED + int(_alpha * 1000))
        _cvar_gap = cvar_abs_gap_windows(_cand, target_real_returns)
        CHOLESKY_SWEEP_ROWS.append({
            "alpha": _alpha,
            "train_corr_mae": round(_mae, 6),
            "train_gaussian_benchmark_mae": round(benchmark_mae, 6),
            "train_mmd_projected": round(float(_mmd), 6),
            "train_cvar_abs_gap": round(float(_cvar_gap), 6),
            "pareto_objective": round(float(_mmd + CHOLESKY_CVAR_OBJECTIVE_WEIGHT * _cvar_gap), 6),
            "passes_train_tolerance": bool(_mae <= benchmark_mae + CORR_MAE_NEAR_GAUSSIAN_TOL),
        })
    cholesky_sweep_df = pd.DataFrame(CHOLESKY_SWEEP_ROWS)
    display(cholesky_sweep_df)
    _passing = [r for r in CHOLESKY_SWEEP_ROWS if r["passes_train_tolerance"]]
    if _passing:
        selected_row = min(_passing, key=lambda r: r["pareto_objective"])
        selected_cholesky_alpha = float(selected_row["alpha"])
        _selection_reason = "pareto_min_mmd_cvar_among_train_corr_passes"
    else:
        selected_row = min(CHOLESKY_SWEEP_ROWS, key=lambda r: (r["train_corr_mae"], r["pareto_objective"]))
        selected_cholesky_alpha = float(selected_row["alpha"])
        _selection_reason = "best_train_corr_mae_none_passed"
    CONFIG["cholesky_alpha_selected"] = selected_cholesky_alpha
    CONFIG["cholesky_alpha_selection_reason"] = _selection_reason
    CONFIG["cholesky_alpha_selection_objective"] = CHOLESKY_SELECTION_OBJECTIVE
    CONFIG["cholesky_alpha_selected_row"] = selected_row
    CONFIG["cholesky_alpha_selection_data"] = "train_only"
    CONFIG["cholesky_sweep_rows"] = CHOLESKY_SWEEP_ROWS
    print(f"Selected Cholesky alpha (train-only Pareto): {selected_cholesky_alpha} ({_selection_reason})")
    synthetic_returns_base = calibrate_asset_correlation_windows(synthetic_returns_base_uncalibrated, target_real_returns, alpha=selected_cholesky_alpha, shrinkage=CHOLESKY_SHRINKAGE)
    synthetic_returns = calibrate_asset_correlation_windows(synthetic_returns_uncalibrated, target_real_returns, alpha=selected_cholesky_alpha, shrinkage=CHOLESKY_SHRINKAGE)
synthetic_returns_base = apply_daily_var5_safety_margin(synthetic_returns_base, target_real_returns, label="crisis_base")
synthetic_returns_raw = synthetic_returns_base_uncalibrated.clone()

# ---- V9.5 unconditional book for exception/tail gates ------------------------
target_factor_norm_all = dataset_factor_windows(train_ds, max_samples=CALIBRATION_TARGET_WINDOWS)
target_factor_raw_all = denormalize_factor_windows(target_factor_norm_all)
target_real_returns_all = dataset_asset_windows(asset_train_ds, max_samples=CALIBRATION_TARGET_WINDOWS)
uncond_factor_raw_uncal, uncond_pool_note = FHS_ENGINE.simulate(N_UNCOND_SCENARIOS, WINDOW_SIZE, None, seed=SEED + 505)
uncond_factor_raw = calibrate_factor_windows(uncond_factor_raw_uncal, target_factor_raw_all)
synthetic_returns_uncond, _ = reconstruct_asset_windows_from_factors(
    uncond_factor_raw, add_residual=True, stress=False, seed=SEED + 606,
    residual_bank=RESIDUAL_WINDOW_BANK_ALL,
)
if APPLY_CHOLESKY_CALIBRATION and len(target_real_returns_all) >= 16:
    synthetic_returns_uncond = calibrate_asset_correlation_windows(
        synthetic_returns_uncond, target_real_returns_all,
        alpha=selected_cholesky_alpha, shrinkage=CHOLESKY_SHRINKAGE,
    )
synthetic_returns_uncond = apply_daily_var5_safety_margin(synthetic_returns_uncond, target_real_returns_all, label="uncond")
CONFIG["uncond_book_scenarios"] = int(len(synthetic_returns_uncond))
CONFIG["uncond_book_pool_note"] = uncond_pool_note
print("Unconditional gate book:", tuple(synthetic_returns_uncond.shape), uncond_pool_note)

real_valid_returns = dataset_asset_windows(asset_valid_ds, max_samples=min(len(asset_valid_ds), N_SCENARIOS))
real_valid_target_returns = dataset_asset_windows(asset_valid_ds, max_samples=min(len(asset_valid_ds), N_SCENARIOS), regime_filter=TARGET_REGIME)
CONFIG["valid_all_windows_sampled"] = int(len(real_valid_returns))
CONFIG["valid_target_regime_windows_sampled"] = int(len(real_valid_target_returns))
CONFIG["cholesky_calibration_applied"] = bool(APPLY_CHOLESKY_CALIBRATION)
CONFIG["cholesky_calibration_alpha"] = float(CHOLESKY_CALIBRATION_ALPHA)
CONFIG["cholesky_shrinkage"] = float(CHOLESKY_SHRINKAGE)
CONFIG["cholesky_calibration_target_source"] = target_source
CONFIG["cholesky_uses_validation"] = False
CONFIG["residual_bootstrap_applied"] = True
CONFIG["stress_scenario_mode_applied"] = bool(STRESS_SCENARIO_MODE)
CONFIG["stress_stratified_sampling_applied"] = bool(STRESS_STRATIFIED_SAMPLING)
CONFIG["stress_multiplier_mean"] = float(np.mean(stress_multipliers))
CONFIG["stress_multiplier_q95"] = float(np.quantile(stress_multipliers, 0.95))
CONFIG["stress_multiplier_counts"] = {str(round(float(k), 4)): int(v) for k, v in zip(*np.unique(stress_multipliers, return_counts=True))}

print("Synthetic stress:", tuple(synthetic_returns.shape), "Synthetic base:", tuple(synthetic_returns_base.shape))
print("Real valid all:", tuple(real_valid_returns.shape), "Real valid target:", tuple(real_valid_target_returns.shape))
print("Synthetic stress summary:", json.dumps(quick_portfolio_summary(synthetic_returns), indent=2))
print("Synthetic base summary:", json.dumps(quick_portfolio_summary(synthetic_returns_base), indent=2))


## 8. Evaluation

Core checks:
- distribution coverage: whether synthetic returns visit real bins;
- correlation fidelity: whether multi-asset dependence survives sampling;
- MMD: kernel distance between real and synthetic windows;
- VaR/CVaR: synthetic portfolio tail risk;
- tail contributors: which assets dominate losses.
        


In [ ]:
#@title Metrics and baselines: dual normal/stress candidates vs train-only baselines
def to_numpy(x):
    return x.detach().cpu().numpy() if torch.is_tensor(x) else np.asarray(x)

def distribution_coverage(real, syn, n_bins=50):
    real = to_numpy(real)
    syn = to_numpy(syn)
    n_assets = real.shape[-1]
    covs = []
    for i in range(n_assets):
        lo, hi = np.nanquantile(real[..., i], [0.001, 0.999])
        if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
            continue
        bins = np.linspace(lo, hi, n_bins)
        rh, _ = np.histogram(real[..., i].ravel(), bins=bins)
        sh, _ = np.histogram(syn[..., i].ravel(), bins=bins)
        total = (rh > 0).sum()
        covs.append(((rh > 0) & (sh > 0)).sum() / max(total, 1))
    return float(np.mean(covs)) if covs else np.nan

def corr_stats(real, syn):
    real = to_numpy(real).reshape(-1, to_numpy(real).shape[-1])
    syn = to_numpy(syn).reshape(-1, to_numpy(syn).shape[-1])
    real_corr = np.nan_to_num(np.corrcoef(real.T), nan=0.0)
    syn_corr = np.nan_to_num(np.corrcoef(syn.T), nan=0.0)
    mask = ~np.eye(real_corr.shape[0], dtype=bool)
    mae = float(np.abs(real_corr[mask] - syn_corr[mask]).mean())
    eig_real = np.sort(np.linalg.eigvalsh(real_corr))[-20:]
    eig_syn = np.sort(np.linalg.eigvalsh(syn_corr))[-20:]
    eig_rmse = float(np.sqrt(np.mean((eig_real - eig_syn) ** 2)))
    return {
        "correlation_fidelity": float(1 - mae),
        "corr_mae": mae,
        "corr_top20_eigen_rmse": eig_rmse,
    }

def mmd_rbf_projected(X, Y, proj_dim=128, max_n=1500, seed=SEED):
    X = to_numpy(X).reshape(to_numpy(X).shape[0], -1)
    Y = to_numpy(Y).reshape(to_numpy(Y).shape[0], -1)
    n = min(max_n, len(X), len(Y))
    X = X[:n].astype(np.float32)
    Y = Y[:n].astype(np.float32)
    rng = np.random.default_rng(seed)
    R = rng.normal(0, 1 / math.sqrt(proj_dim), size=(X.shape[1], proj_dim)).astype(np.float32)
    Xp = X @ R
    Yp = Y @ R
    mu = Xp.mean(axis=0, keepdims=True)
    sigma = Xp.std(axis=0, keepdims=True) + 1e-6
    Xp = (Xp - mu) / sigma
    Yp = (Yp - mu) / sigma

    def sqdist(A, B):
        return np.maximum((A * A).sum(1, keepdims=True) + (B * B).sum(1)[None, :] - 2 * A @ B.T, 0)

    Z = np.vstack([Xp[: min(300, len(Xp))], Yp[: min(300, len(Yp))]])
    d_med = sqdist(Z, Z)
    med = np.median(d_med[d_med > 0])
    gamma = 1.0 / max(float(med), 1e-6)
    return float(np.exp(-gamma * sqdist(Xp, Xp)).mean() + np.exp(-gamma * sqdist(Yp, Yp)).mean() - 2 * np.exp(-gamma * sqdist(Xp, Yp)).mean())

def parse_int_list(text):
    return [int(x.strip()) for x in str(text).split(",") if x.strip()]

def mmd_rbf_projected_multi(X, Y, seeds=None, proj_dim=128, max_n=1500):
    seeds = parse_int_list(MMD_STABILITY_SEEDS) if seeds is None else list(seeds)
    vals = [
        mmd_rbf_projected(X, Y, proj_dim=proj_dim, max_n=max_n, seed=int(seed))
        for seed in seeds
    ]
    return {
        "mmd_rbf_projected_multi_mean": float(np.mean(vals)),
        "mmd_rbf_projected_multi_std": float(np.std(vals)),
        "mmd_rbf_projected_multi_min": float(np.min(vals)),
        "mmd_rbf_projected_multi_max": float(np.max(vals)),
    }

def sample_windows_np(x, n, seed=PRIMARY_EVAL_SUBSAMPLE_SEED):
    arr = to_numpy(x)
    if len(arr) <= n:
        return arr
    rng = np.random.default_rng(int(seed))
    idx = rng.choice(len(arr), size=int(n), replace=False)
    return arr[idx]

def portfolio_terminal_returns(samples, weights=None):
    arr = to_numpy(samples)
    if weights is None:
        weights = np.ones(arr.shape[-1]) / arr.shape[-1]
    daily = np.clip(arr @ weights, -0.95, 1.50)
    return np.prod(1 + daily, axis=1) - 1, daily

def max_drawdown(daily_returns):
    wealth = np.cumprod(1 + daily_returns, axis=1)
    peaks = np.maximum.accumulate(wealth, axis=1)
    dd = wealth / peaks - 1
    return dd.min(axis=1)

def historical_bootstrap_windows(reference, n_samples, seed=SEED):
    arr = to_numpy(reference)
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(arr), size=n_samples, replace=True)
    return arr[idx]

def strided_windows_np(windows, stride=WINDOW_SIZE):
    arr = to_numpy(windows)
    stride = max(1, int(stride))
    return arr[::stride]

def gaussian_cov_from_windows(reference, n_samples, window_size, seed=SEED):
    rng = np.random.default_rng(seed)
    ref = to_numpy(reference).astype(np.float32)
    X = ref.reshape(-1, ref.shape[-1])
    mu = X.mean(axis=0)
    cov = np.cov(X.T)
    diag = np.diag(np.diag(cov))
    cov = 0.95 * cov + 0.05 * diag
    jitter = 1e-6 * np.eye(cov.shape[0])
    L = np.linalg.cholesky(cov + jitter)
    z = rng.normal(size=(n_samples, window_size, ref.shape[-1])).astype(np.float32)
    return z @ L.T + mu

def t_copula_from_windows(reference, n_samples, window_size, df=BASELINE_T_COPULA_DF, seed=SEED):
    rng = np.random.default_rng(seed)
    ref = to_numpy(reference).astype(np.float32)
    X = ref.reshape(-1, ref.shape[-1])
    mu = X.mean(axis=0)
    cov = np.cov(X.T)
    diag = np.diag(np.diag(cov))
    cov = 0.95 * cov + 0.05 * diag
    L = np.linalg.cholesky(cov + 1e-6 * np.eye(cov.shape[0]))
    z = rng.normal(size=(n_samples, window_size, ref.shape[-1])).astype(np.float32)
    chi = rng.chisquare(df, size=(n_samples, window_size, 1)).astype(np.float32)
    return (z @ L.T) / np.sqrt(np.maximum(chi / df, 1e-6)) + mu

def filtered_historical_simulation(reference, n_samples, seed=SEED):
    rng = np.random.default_rng(seed)
    ref = to_numpy(reference).astype(np.float32)
    flat = ref.reshape(-1, ref.shape[-1])
    lam = float(FHS_EWMA_LAMBDA)
    vol = np.zeros_like(flat)
    vol[0] = np.nanstd(flat, axis=0) + 1e-6
    for i in range(1, len(flat)):
        vol[i] = np.sqrt(lam * vol[i - 1] ** 2 + (1 - lam) * flat[i - 1] ** 2) + 1e-6
    resid = flat / vol
    idx = rng.choice(len(resid), size=n_samples * ref.shape[1], replace=True)
    sampled = resid[idx].reshape(n_samples, ref.shape[1], ref.shape[2])
    vol_idx = rng.choice(len(vol), size=n_samples * ref.shape[1], replace=True)
    sampled_vol = vol[vol_idx].reshape(n_samples, ref.shape[1], ref.shape[2])
    return sampled * sampled_vol

def gaussian_factor_same_stack(n_samples, seed=SEED):
    raw = np.asarray(target_factor_raw, dtype=np.float32)
    flat = raw.reshape(-1, raw.shape[-1])
    rng = np.random.default_rng(seed)
    mu = flat.mean(axis=0)
    cov = np.cov(flat.T)
    diag = np.diag(np.diag(cov))
    cov = 0.95 * cov + 0.05 * diag
    L = np.linalg.cholesky(cov + 1e-6 * np.eye(cov.shape[0]))
    z = rng.normal(size=(n_samples, WINDOW_SIZE, raw.shape[-1])).astype(np.float32)
    factor_raw = z @ L.T + mu
    factor_raw = calibrate_factor_windows(factor_raw, target_factor_raw)
    returns, _ = reconstruct_asset_windows_from_factors(factor_raw, add_residual=True, stress=False, seed=seed + 17)
    return to_numpy(returns)

def metric_bundle(name, samples, real_ref):
    weights = np.ones(to_numpy(samples).shape[-1]) / to_numpy(samples).shape[-1]
    port, daily = portfolio_terminal_returns(samples, weights)
    var5 = np.quantile(port, 0.05)
    var1 = np.quantile(port, 0.01)
    cvar5 = port[port <= var5].mean()
    dd = max_drawdown(daily)
    out = {
        "model": name,
        "distribution_coverage": distribution_coverage(real_ref, samples),
        "mmd_rbf_projected": mmd_rbf_projected(real_ref, samples, proj_dim=MMD_PROJECTION_DIM, max_n=MMD_MAX_WINDOWS),
        "terminal_mean": float(np.mean(port)),
        "terminal_median": float(np.median(port)),
        "var5": float(var5),
        "var1": float(var1),
        "cvar5": float(cvar5),
        "probability_loss": float((port < 0).mean()),
        "probability_drawdown_10pct": float((dd <= -0.10).mean()),
    }
    out.update(corr_stats(real_ref, samples))
    return out

weights = np.ones(N_ASSETS_ACTUAL) / N_ASSETS_ACTUAL
train_all_reference = dataset_asset_windows(asset_train_ds, max_samples=min(len(asset_train_ds), BASELINE_SCENARIOS), seed=SEED + 3)
train_target_reference = target_real_returns if len(target_real_returns) >= 64 else train_all_reference

if len(real_valid_target_returns) >= 64:
    real_eval_returns = real_valid_target_returns
    eval_reference_name = "valid_target_regime"
    baseline_train_reference = train_target_reference
else:
    real_eval_returns = real_valid_returns
    eval_reference_name = "valid_all_regimes_fallback"
    baseline_train_reference = train_all_reference

MIN_ROBUST_EVAL_WINDOWS = 64
ABS_MIN_EVAL_WINDOWS = 16

strict_nonoverlap_eval = strided_windows_np(real_eval_returns, WINDOW_SIZE)
CONFIG["eval_nonoverlap_windows_available"] = int(len(strict_nonoverlap_eval))
CONFIG["eval_nonoverlap_min_required"] = int(MIN_ROBUST_EVAL_WINDOWS)
CONFIG["eval_nonoverlap_gate_ok"] = bool(len(strict_nonoverlap_eval) >= MIN_ROBUST_EVAL_WINDOWS)

real_eval_returns_for_metrics = strided_windows_np(real_eval_returns, EVAL_WINDOW_STRIDE)
metric_stride = int(EVAL_WINDOW_STRIDE)

if len(real_eval_returns_for_metrics) < MIN_ROBUST_EVAL_WINDOWS:
    stride_candidates = [
        max(1, WINDOW_SIZE // 2),
        max(1, WINDOW_SIZE // 3),
        max(1, WINDOW_SIZE // 5),
        max(1, WINDOW_SIZE // 10),
        1,
    ]

    best_stride = metric_stride
    best_reference = real_eval_returns_for_metrics

    for candidate_stride in stride_candidates:
        candidate_reference = strided_windows_np(real_eval_returns, candidate_stride)
        if len(candidate_reference) > len(best_reference):
            best_stride = int(candidate_stride)
            best_reference = candidate_reference
        if len(candidate_reference) >= MIN_ROBUST_EVAL_WINDOWS:
            break

    real_eval_returns_for_metrics = best_reference
    metric_stride = best_stride

CONFIG["eval_window_stride_applied"] = int(metric_stride)
CONFIG["eval_windows_are_overlapping"] = bool(metric_stride < WINDOW_SIZE)
CONFIG["eval_metric_windows_available"] = int(len(real_eval_returns_for_metrics))
CONFIG["eval_metric_reference_note"] = (
    "non_overlapping_headline"
    if metric_stride >= WINDOW_SIZE
    else "adaptive_stride_diagnostics_only_nonoverlap_gate_fails"
)

n_base = min(
    BASELINE_SCENARIOS,
    len(real_eval_returns_for_metrics),
    len(synthetic_returns),
    len(synthetic_returns_base),
    len(baseline_train_reference),
)
CONFIG["eval_metric_windows_used"] = int(n_base)
CONFIG["eval_metric_sample_size_ok"] = bool(n_base >= MIN_ROBUST_EVAL_WINDOWS)

if n_base < ABS_MIN_EVAL_WINDOWS:
    raise RuntimeError(f"Too few evaluation windows even for diagnostic metrics: {n_base}")

if n_base < MIN_ROBUST_EVAL_WINDOWS:
    print(f"WARNING: only {n_base} eval windows available. Continuing for diagnostics; endpoint promotion gate must fail.")

bootstrap_returns = historical_bootstrap_windows(baseline_train_reference, n_base, seed=SEED + 7)
gaussian_returns = gaussian_cov_from_windows(baseline_train_reference, n_base, WINDOW_SIZE, seed=SEED + 11)
t_copula_returns = t_copula_from_windows(baseline_train_reference, n_base, WINDOW_SIZE, seed=SEED + 13)
fhs_returns = filtered_historical_simulation(baseline_train_reference, n_base, seed=SEED + 19)
same_stack_gaussian_returns = gaussian_factor_same_stack(n_base, seed=SEED + 23)

model_samples = {
    STRESS_CANDIDATE_MODEL: sample_windows_np(synthetic_returns, n_base, PRIMARY_EVAL_SUBSAMPLE_SEED),
    NORMAL_CANDIDATE_MODEL: sample_windows_np(synthetic_returns_base, n_base, PRIMARY_EVAL_SUBSAMPLE_SEED),
    "historical_bootstrap_train": bootstrap_returns,
    "gaussian_cov_train": gaussian_returns,
    "t_copula_train": t_copula_returns,
    "filtered_historical_simulation_train": fhs_returns,
    "gaussian_factor_same_calibration_stack": same_stack_gaussian_returns,
}
if "synthetic_returns_uncalibrated" in globals():
    model_samples["fhs_v9_stress_uncalibrated"] = sample_windows_np(synthetic_returns_uncalibrated, n_base, PRIMARY_EVAL_SUBSAMPLE_SEED)
if "synthetic_returns_base_uncalibrated" in globals():
    model_samples["fhs_v9_base_uncalibrated"] = sample_windows_np(synthetic_returns_base_uncalibrated, n_base, PRIMARY_EVAL_SUBSAMPLE_SEED)
CONFIG["eval_uses_synthetic_subsample"] = True
CONFIG["primary_eval_subsample_seed"] = int(PRIMARY_EVAL_SUBSAMPLE_SEED)

metrics_table = pd.DataFrame(
    [metric_bundle(name, samples, real_eval_returns_for_metrics[:n_base]) for name, samples in model_samples.items()]
).set_index("model")

mmd_stability_rows = []
for name, samples in model_samples.items():
    row = {"model": name}
    row.update(mmd_rbf_projected_multi(real_eval_returns_for_metrics[:n_base], samples, proj_dim=MMD_PROJECTION_DIM, max_n=MMD_MAX_WINDOWS))
    mmd_stability_rows.append(row)
mmd_stability_table = pd.DataFrame(mmd_stability_rows).set_index("model")
for col in mmd_stability_table.columns:
    metrics_table[col] = mmd_stability_table[col]

normal_candidate_model = NORMAL_CANDIDATE_MODEL
stress_candidate_model = STRESS_CANDIDATE_MODEL
candidate_model = PRODUCT_CANDIDATE_MODEL
stress_port, stress_daily = portfolio_terminal_returns(synthetic_returns, weights)
base_port, base_daily = portfolio_terminal_returns(synthetic_returns_base, weights)
syn_port, syn_daily = stress_port, stress_daily  # stress diagnostics and walk-forward coverage use the adverse book
real_port, real_daily = portfolio_terminal_returns(real_eval_returns, weights)
real_valid_port_all, real_valid_daily_all = portfolio_terminal_returns(real_valid_returns, weights)

train_target_metrics = pd.Series(quick_portfolio_summary(train_target_reference), name="train_target_empirical")
valid_eval_metrics = pd.Series(quick_portfolio_summary(real_eval_returns), name=eval_reference_name)
synthetic_stress_metrics = pd.Series(quick_portfolio_summary(synthetic_returns), name="fhs_v9_stress")
synthetic_base_metrics = pd.Series(quick_portfolio_summary(synthetic_returns_base), name="fhs_v9_base")
candidate_metrics_series = synthetic_stress_metrics if candidate_model == stress_candidate_model else synthetic_base_metrics
regime_calibration = pd.concat([train_target_metrics, valid_eval_metrics, synthetic_stress_metrics, synthetic_base_metrics], axis=1)

metrics = metrics_table.loc[candidate_model].to_dict()
metrics.update({f"train_target_{k}": float(v) for k, v in train_target_metrics.items()})
metrics.update({f"eval_reference_{k}": float(v) for k, v in valid_eval_metrics.items()})
metrics.update({f"synthetic_{k}": float(v) for k, v in candidate_metrics_series.items()})
metrics.update({f"stress_{k}": float(v) for k, v in synthetic_stress_metrics.items()})
metrics["candidate_model"] = candidate_model
metrics["normal_candidate_model"] = normal_candidate_model
metrics["stress_candidate_model"] = stress_candidate_model
metrics["stress_model_for_walk_forward"] = stress_candidate_model
metrics["selected_guidance_scale"] = float(CONFIG.get("selected_guidance_scale", 1.0))
metrics["eval_reference_name"] = eval_reference_name
metrics["n_eval_windows"] = int(n_base)
metrics["n_valid_target_windows_available"] = int(len(real_valid_target_returns))
CONFIG["candidate_model"] = candidate_model
CONFIG["normal_candidate_model"] = normal_candidate_model
CONFIG["stress_candidate_model"] = stress_candidate_model
CONFIG["stress_model_for_walk_forward"] = stress_candidate_model
CONFIG["eval_reference_name"] = eval_reference_name
CONFIG["baselines_use_validation_for_generation"] = False
CONFIG["mmd_standardization"] = "real_reference_only"

def period_portfolio_terminals(returns_frame, start, end, columns, window_size):
    period = returns_frame.loc[pd.Timestamp(start):pd.Timestamp(end), list(columns)].dropna(how="any")
    if len(period) < max(5, window_size // 2):
        return np.array([])
    arr = period.values.astype(np.float32)
    if len(arr) < window_size:
        windows = arr[None, :, :]
    else:
        windows = np.stack([arr[i:i + window_size] for i in range(0, len(arr) - window_size + 1)])
    weights = np.ones(windows.shape[-1]) / windows.shape[-1]
    daily = np.clip(windows @ weights, -0.95, 1.50)
    return np.prod(1 + daily, axis=1) - 1

def walk_forward_crisis_audit(synthetic_port):
    crisis_periods = {
        "covid_crash_2020": ("2020-02-18", "2020-04-30"),
        "inflation_bear_2022": ("2022-01-03", "2022-10-31"),
        "bank_stress_2023": ("2023-03-01", "2023-04-30"),
    }
    rows = []
    syn_q01, syn_q05, syn_q50 = np.quantile(synthetic_port, [0.01, 0.05, 0.50])
    for name, (start, end) in crisis_periods.items():
        actual = period_portfolio_terminals(returns_df, start, end, train_ds.columns, WINDOW_SIZE)
        if len(actual) == 0:
            continue
        rows.append({
            "period": name,
            "start": start,
            "end": end,
            "n_actual_windows": int(len(actual)),
            "actual_min_terminal": float(np.min(actual)),
            "actual_median_terminal": float(np.median(actual)),
            "actual_mean_terminal": float(np.mean(actual)),
            "synthetic_q01": float(syn_q01),
            "synthetic_q05": float(syn_q05),
            "synthetic_median": float(syn_q50),
            "crisis_min_covered_by_syn_1pct": bool(syn_q01 <= np.min(actual)),
            "crisis_min_covered_by_syn_5pct": bool(syn_q05 <= np.min(actual)),
        })
    return pd.DataFrame(rows)

walk_forward_df = walk_forward_crisis_audit(stress_port)
metrics["walk_forward_periods"] = int(len(walk_forward_df))
metrics["walk_forward_1pct_coverage_rate"] = float(walk_forward_df["crisis_min_covered_by_syn_1pct"].mean()) if len(walk_forward_df) else np.nan
metrics["walk_forward_5pct_coverage_rate"] = float(walk_forward_df["crisis_min_covered_by_syn_5pct"].mean()) if len(walk_forward_df) else np.nan

gaussian_row = metrics_table.loc["gaussian_cov_train"]
candidate_row = metrics_table.loc[candidate_model]
relative_to_gaussian = pd.Series({
    "mmd_ratio_candidate_vs_gaussian": float(candidate_row["mmd_rbf_projected"] / max(gaussian_row["mmd_rbf_projected"], 1e-12)),
    "mmd_multi_ratio_candidate_vs_gaussian": float(candidate_row["mmd_rbf_projected_multi_mean"] / max(gaussian_row["mmd_rbf_projected_multi_mean"], 1e-12)),
    "corr_mae_delta_candidate_minus_gaussian": float(candidate_row["corr_mae"] - gaussian_row["corr_mae"]),
    "eigen_rmse_delta_candidate_minus_gaussian": float(candidate_row["corr_top20_eigen_rmse"] - gaussian_row["corr_top20_eigen_rmse"]),
    "candidate_corr_near_gaussian_within_tol": bool(candidate_row["corr_mae"] <= gaussian_row["corr_mae"] + CORR_MAE_NEAR_GAUSSIAN_TOL),
    "candidate_mmd_ratio_within_research_gate": bool(candidate_row["mmd_rbf_projected_multi_mean"] <= gaussian_row["mmd_rbf_projected_multi_mean"] * MMD_RATIO_MAX_RESEARCH),
})
metrics.update({k: (bool(v) if isinstance(v, (bool, np.bool_)) else float(v)) for k, v in relative_to_gaussian.items()})

calibration_audit_table = metrics_table.loc[
    [idx for idx in ["fhs_v9_base_uncalibrated", "fhs_v9_base", "fhs_v9_stress_uncalibrated", "fhs_v9_stress"] if idx in metrics_table.index],
    ["mmd_rbf_projected", "mmd_rbf_projected_multi_mean", "corr_mae", "corr_top20_eigen_rmse", "cvar5", "probability_drawdown_10pct"],
].copy()

subsample_rows = []
for seed in parse_int_list(EVAL_SYNTHETIC_SUBSAMPLE_SEEDS):
    base_subset = sample_windows_np(synthetic_returns_base, n_base, seed)
    stress_subset = sample_windows_np(synthetic_returns, n_base, seed)
    row = {
        "subsample_seed": int(seed),
        "base_mmd": mmd_rbf_projected(real_eval_returns_for_metrics[:n_base], base_subset, proj_dim=MMD_PROJECTION_DIM, max_n=MMD_MAX_WINDOWS, seed=seed),
        "base_mmd_multi_mean": mmd_rbf_projected_multi(real_eval_returns_for_metrics[:n_base], base_subset, proj_dim=MMD_PROJECTION_DIM, max_n=MMD_MAX_WINDOWS)["mmd_rbf_projected_multi_mean"],
        "base_corr_mae": corr_stats(real_eval_returns_for_metrics[:n_base], base_subset)["corr_mae"],
        "stress_mmd": mmd_rbf_projected(real_eval_returns_for_metrics[:n_base], stress_subset, proj_dim=MMD_PROJECTION_DIM, max_n=MMD_MAX_WINDOWS, seed=seed),
        "stress_q01": float(np.quantile(portfolio_terminal_returns(stress_subset, weights)[0], 0.01)),
    }
    subsample_rows.append(row)
synthetic_subsample_sensitivity = pd.DataFrame(subsample_rows)
metrics["subsample_base_mmd_multi_mean_mean"] = float(synthetic_subsample_sensitivity["base_mmd_multi_mean"].mean())
metrics["subsample_base_mmd_multi_mean_std"] = float(synthetic_subsample_sensitivity["base_mmd_multi_mean"].std())
metrics["subsample_base_corr_mae_mean"] = float(synthetic_subsample_sensitivity["base_corr_mae"].mean())
CONFIG["eval_synthetic_subsample_seeds"] = EVAL_SYNTHETIC_SUBSAMPLE_SEEDS

real_eval_factor_norm = dataset_factor_windows(
    valid_ds,
    max_samples=min(len(valid_ds), n_base),
    regime_filter=TARGET_REGIME if eval_reference_name == "valid_target_regime" else None,
)
if len(real_eval_factor_norm) < 64:
    real_eval_factor_norm = dataset_factor_windows(valid_ds, max_samples=min(len(valid_ds), n_base))
real_eval_factor_raw = denormalize_factor_windows(real_eval_factor_norm)
factor_n = min(len(real_eval_factor_raw), len(synthetic_factor_raw), n_base)
stress_factor_raw_for_metrics = stress_ladder(np.asarray(synthetic_factor_raw[:factor_n], dtype=np.float32), seed=SEED + 202)[0]
factor_space_metrics = pd.Series({
    "factor_mmd_base": mmd_rbf_projected(real_eval_factor_raw[:factor_n], synthetic_factor_raw[:factor_n], proj_dim=min(MMD_PROJECTION_DIM, N_FACTORS_ACTUAL), max_n=factor_n),
    "factor_mmd_base_multi_mean": mmd_rbf_projected_multi(real_eval_factor_raw[:factor_n], synthetic_factor_raw[:factor_n], proj_dim=min(MMD_PROJECTION_DIM, N_FACTORS_ACTUAL), max_n=factor_n)["mmd_rbf_projected_multi_mean"],
    "factor_mmd_stress": mmd_rbf_projected(real_eval_factor_raw[:factor_n], stress_factor_raw_for_metrics, proj_dim=min(MMD_PROJECTION_DIM, N_FACTORS_ACTUAL), max_n=factor_n),
    "factor_mmd_stress_multi_mean": mmd_rbf_projected_multi(real_eval_factor_raw[:factor_n], stress_factor_raw_for_metrics, proj_dim=min(MMD_PROJECTION_DIM, N_FACTORS_ACTUAL), max_n=factor_n)["mmd_rbf_projected_multi_mean"],
    "factor_eval_windows": int(factor_n),
}, name="value")
metrics.update({k: float(v) for k, v in factor_space_metrics.items()})

stress_multiplier_counts = {str(float(k)): int(v) for k, v in zip(*np.unique(stress_multipliers, return_counts=True))}
stress_q = float(np.quantile(stress_port, ENDPOINT_STRESS_QUANTILE))
endpoint_contract = {
    "endpoint": "/api/v1/workspaces/{workspaceId}/market-simulation",
    "status": "stress_candidate_offline_only",
    "base_model": normal_candidate_model,
    "stress_model": stress_candidate_model,
    "product_candidate_model": candidate_model,
    "generated_scenarios": int(len(stress_port)),
    "endpoint_default_scenarios": int(ENDPOINT_DEFAULT_SCENARIOS),
    "endpoint_min_scenarios": int(ENDPOINT_MIN_SCENARIOS),
    "endpoint_min_stress_scenarios": int(ENDPOINT_MIN_STRESS_SCENARIOS),
    "endpoint_stress_quantile": float(ENDPOINT_STRESS_QUANTILE),
    "endpoint_small_request_policy": ENDPOINT_SMALL_REQUEST_POLICY,
    "stress_stratified_sampling": bool(STRESS_STRATIFIED_SAMPLING),
    "stress_multiplier_counts": json.dumps(stress_multiplier_counts, sort_keys=True),
    "stress_book_q01": stress_q,
    "scenario_count_ok_for_stress_endpoint": bool(len(stress_port) >= ENDPOINT_MIN_STRESS_SCENARIOS),
    "ready_for_endpoint_requires": "stress candidate must pass stress scorecard, coverage gates, train-only correlation, and endpoint count",
}
endpoint_contract_table = pd.Series(endpoint_contract, name="value").to_frame()
metrics["endpoint_scenario_count_ok"] = bool(endpoint_contract["scenario_count_ok_for_stress_endpoint"])
metrics["endpoint_stress_book_q01"] = float(stress_q)
CONFIG["endpoint_contract_status"] = endpoint_contract["status"]
CONFIG["endpoint_contract"] = endpoint_contract

print("Evaluation reference:", eval_reference_name, "| windows:", n_base, "| product candidate:", candidate_model, "| normal book:", normal_candidate_model)
display(metrics_table)

def make_portfolio_weight_suite(n_assets, seed=SEED):
    rng = np.random.default_rng(seed)
    rows = [("equal_weight", np.ones(n_assets) / n_assets)]
    for i in range(PORTFOLIO_TEST_COUNT):
        rows.append((f"random_dirichlet_{i+1:02d}", rng.dirichlet(np.ones(n_assets))))
    for level in parse_float_list(PORTFOLIO_CONCENTRATION_LEVELS):
        w = np.ones(n_assets) * ((1 - level) / max(n_assets - 1, 1))
        w[int(rng.integers(0, n_assets))] = level
        rows.append((f"single_name_{float(level):.0%}", w / w.sum()))
    return rows

portfolio_suite_rows = []
for portfolio_name, portfolio_weights in make_portfolio_weight_suite(N_ASSETS_ACTUAL):
    for model_name, samples in model_samples.items():
        port, daily = portfolio_terminal_returns(samples, portfolio_weights)
        var5_suite = np.quantile(port, 0.05)
        var1_suite = np.quantile(port, 0.01)
        portfolio_suite_rows.append({
            "portfolio": portfolio_name,
            "model": model_name,
            "var5": float(var5_suite),
            "var1": float(var1_suite),
            "cvar5": float(port[port <= var5_suite].mean()),
            "probability_drawdown_10pct": float((max_drawdown(daily) <= -0.10).mean()),
        })
portfolio_suite_table = pd.DataFrame(portfolio_suite_rows)
display(portfolio_suite_table.head(20))
display(metrics_table.T)
display(regime_calibration)
display(walk_forward_df)
display(relative_to_gaussian.to_frame("value"))
display(calibration_audit_table)
display(synthetic_subsample_sensitivity)
display(factor_space_metrics.to_frame())
display(endpoint_contract_table)


In [ ]:
#@title V9.7 gates: stress coverage + conditional rolling VaR backtest
from scipy import stats as sp_stats

EW_WEIGHTS = np.ones(N_ASSETS_ACTUAL) / N_ASSETS_ACTUAL

def pooled_daily_portfolio(x):
    arr = to_numpy(x)
    return np.clip(arr @ EW_WEIGHTS, -0.95, 1.50).reshape(-1)

def kupiec_pof(n_obs, n_exceed, p):
    n_obs = int(n_obs); x = int(n_exceed)
    if n_obs == 0:
        return {"n": 0, "exceed": 0, "expected": 0.0, "lr": np.nan, "p_value": np.nan, "ok": False}
    pi_hat = min(max(x / n_obs, 1e-10), 1 - 1e-10)
    p = min(max(p, 1e-10), 1 - 1e-10)
    ll_null = (n_obs - x) * np.log(1 - p) + x * np.log(p)
    ll_alt = (n_obs - x) * np.log(1 - pi_hat) + x * np.log(pi_hat)
    lr = -2.0 * (ll_null - ll_alt)
    p_value = float(sp_stats.chi2.sf(lr, df=1))
    return {"n": n_obs, "exceed": x, "expected": float(n_obs * p), "lr": float(lr), "p_value": p_value,
            "ok": bool(p_value >= KUPIEC_MIN_P)}

def christoffersen_independence(hits):
    hits = np.asarray(hits, dtype=int)
    if len(hits) < 3:
        return {"n00": 0, "n01": 0, "n10": 0, "n11": 0, "lr": np.nan, "p_value": np.nan, "ok": False}
    prev, curr = hits[:-1], hits[1:]
    n00 = int(((prev == 0) & (curr == 0)).sum())
    n01 = int(((prev == 0) & (curr == 1)).sum())
    n10 = int(((prev == 1) & (curr == 0)).sum())
    n11 = int(((prev == 1) & (curr == 1)).sum())
    eps = 1e-10
    pi = min(max((n01 + n11) / max(n00 + n01 + n10 + n11, 1), eps), 1 - eps)
    p01 = min(max(n01 / max(n00 + n01, 1), eps), 1 - eps)
    p11 = min(max(n11 / max(n10 + n11, 1), eps), 1 - eps)
    ll_null = (n00 + n10) * np.log(1 - pi) + (n01 + n11) * np.log(pi)
    ll_alt = n00 * np.log(1 - p01) + n01 * np.log(p01) + n10 * np.log(1 - p11) + n11 * np.log(p11)
    lr = -2.0 * (ll_null - ll_alt)
    p_value = float(sp_stats.chi2.sf(lr, df=1))
    return {"n00": n00, "n01": n01, "n10": n10, "n11": n11, "lr": float(lr), "p_value": p_value,
            "ok": bool(p_value >= CHRISTOFFERSEN_MIN_P)}

def christoffersen_conditional_coverage(kupiec_result, independence_result):
    lr = float(kupiec_result.get("lr", np.nan)) + float(independence_result.get("lr", np.nan))
    p_value = float(sp_stats.chi2.sf(lr, df=2)) if np.isfinite(lr) else np.nan
    return {"lr": lr, "p_value": p_value, "ok": bool(np.isfinite(p_value) and p_value >= CHRISTOFFERSEN_MIN_P)}

def ewma_past_vol(returns, lam=0.94, initial_var=None):
    r = np.asarray(returns, dtype=np.float64)
    warm = r[:min(max(60, WINDOW_SIZE), len(r))]
    var = float(initial_var) if initial_var is not None else float(np.nanvar(warm))
    var = max(var, 1e-10)
    sigma = np.empty(len(r), dtype=np.float64)
    for i, value in enumerate(r):
        sigma[i] = np.sqrt(max(var, 1e-10))
        var = float(lam * var + (1.0 - lam) * value * value)
    return sigma, var

def rolling_conditional_var_backtest(train_returns, valid_returns, q=0.05, lam=0.94):
    train_returns = np.asarray(train_returns, dtype=np.float64)
    valid_returns = np.asarray(valid_returns, dtype=np.float64)
    sigma_train, last_var = ewma_past_vol(train_returns, lam=lam)
    start = min(max(60, WINDOW_SIZE), len(train_returns) - 1)
    innovations = train_returns[start:] / np.maximum(sigma_train[start:], 1e-8)
    innovation_q = float(np.quantile(innovations, q))
    sigma_valid, _ = ewma_past_vol(valid_returns, lam=lam, initial_var=last_var)
    var_line = sigma_valid * innovation_q
    hits = (valid_returns <= var_line).astype(int)
    kupiec = kupiec_pof(len(valid_returns), int(hits.sum()), q)
    independence = christoffersen_independence(hits)
    conditional = christoffersen_conditional_coverage(kupiec, independence)
    return {
        "q": float(q),
        "lambda": float(lam),
        "innovation_quantile": innovation_q,
        "train_innovation_count": int(len(innovations)),
        "validation_days": int(len(valid_returns)),
        "exceedances": int(hits.sum()),
        "expected_exceedances": float(len(valid_returns) * q),
        "kupiec": kupiec,
        "christoffersen_independence": independence,
        "christoffersen_conditional_coverage": conditional,
        "ok": bool(kupiec["ok"] and independence["ok"] and conditional["ok"]),
    }

def hill_index(losses, tail_fraction=0.05, min_k=20):
    x = np.sort(np.asarray(losses, dtype=np.float64))
    x = -x[x < 0]
    x = np.sort(x)[::-1]
    k = max(min_k, int(len(x) * tail_fraction))
    k = min(k, len(x) - 1)
    if k < 5:
        return np.nan
    top = x[: k + 1]
    return float(np.mean(np.log(top[:-1] / max(top[-1], 1e-12))))

# --- Validation day pools, split by regime (labels from the train-only HMM)
valid_daily_frame = returns_aligned.loc[valid_label_start:, asset_columns]
valid_daily_regimes = regime_series.reindex(valid_daily_frame.index).ffill().astype(int)
valid_daily_port = np.clip(valid_daily_frame.values.astype(np.float64) @ EW_WEIGHTS, -0.95, 1.50)
crisis_mask = (valid_daily_regimes == int(TARGET_REGIME)).values
crisis_daily_port = valid_daily_port[crisis_mask]
train_daily_frame = returns_aligned.loc[returns_aligned.index < pd.Timestamp(valid_label_start), asset_columns]
train_daily_port = np.clip(train_daily_frame.values.astype(np.float64) @ EW_WEIGHTS, -0.95, 1.50)
CONFIG["v91_valid_days_all"] = int(len(valid_daily_port))
CONFIG["v91_valid_days_crisis"] = int(crisis_mask.sum())

# --- Diagnostic only: static scenario-book exception tests.
uncond_daily_pool = pooled_daily_portfolio(synthetic_returns_uncond)
q05_uncond = float(np.quantile(uncond_daily_pool, 0.05))
q01_uncond = float(np.quantile(uncond_daily_pool, 0.01))
kupiec_uncond_var5 = kupiec_pof(len(valid_daily_port), int((valid_daily_port <= q05_uncond).sum()), 0.05)
kupiec_uncond_var1 = kupiec_pof(len(valid_daily_port), int((valid_daily_port <= q01_uncond).sum()), 0.01)
print("Static unconditional book VaR5 diagnostic:", kupiec_uncond_var5)
print("Static unconditional book VaR1 diagnostic:", kupiec_uncond_var1)

base_daily_pool = pooled_daily_portfolio(synthetic_returns_base)
q05_crisis = float(np.quantile(base_daily_pool, 0.05))
q01_crisis = float(np.quantile(base_daily_pool, 0.01))
kupiec_crisis_var5 = kupiec_pof(len(crisis_daily_port), int((crisis_daily_port <= q05_crisis).sum()), 0.05)
kupiec_crisis_var1 = kupiec_pof(len(crisis_daily_port), int((crisis_daily_port <= q01_crisis).sum()), 0.01)
print("Static crisis-base VaR5 diagnostic:", kupiec_crisis_var5)
print("Static crisis-base VaR1 diagnostic:", kupiec_crisis_var1)

# --- V9.7 gate: conditional rolling VaR, train innovations + observable rolling volatility.
conditional_var_backtests = {}
for q in [float(x) for x in str(CONDITIONAL_VAR_QUANTILES).split(",") if x.strip()]:
    conditional_var_backtests[f"var{int(round(q * 100))}"] = rolling_conditional_var_backtest(
        train_daily_port,
        valid_daily_port,
        q=q,
        lam=float(CONDITIONAL_VAR_EWMA_LAMBDA),
    )
print("Conditional rolling VaR backtests:", json.dumps(conditional_var_backtests, indent=2))

# --- Diagnostic tail and per-asset checks for the static book.
hill_uncond = hill_index(uncond_daily_pool)
hill_valid_all = hill_index(valid_daily_port)
hill_uncond_rel_err = abs(hill_uncond - hill_valid_all) / max(abs(hill_valid_all), 1e-6) if np.isfinite(hill_uncond) and np.isfinite(hill_valid_all) else np.inf
hill_crisis = hill_index(base_daily_pool)
hill_valid_crisis = hill_index(crisis_daily_port)
print(f"Hill static uncond {hill_uncond:.4f} vs all-valid {hill_valid_all:.4f} (rel err {hill_uncond_rel_err:.1%}) [diagnostic]")
print(f"Hill crisis-base {hill_crisis:.4f} vs crisis-valid {hill_valid_crisis:.4f} [diagnostic]")

uncond_flat_assets = to_numpy(synthetic_returns_uncond).reshape(-1, N_ASSETS_ACTUAL)
q05_syn_assets = np.quantile(uncond_flat_assets, 0.05, axis=0)
q05_real_assets = np.quantile(valid_daily_frame.values.astype(np.float64), 0.05, axis=0)
per_asset_rel = np.abs(q05_syn_assets - q05_real_assets) / np.maximum(np.abs(q05_real_assets), 1e-4)
per_asset_q05_median_rel = float(np.median(per_asset_rel))
print(f"Per-asset daily q05 median relative error (static uncond vs all days): {per_asset_q05_median_rel:.2%} [diagnostic]")

# --- Walk-forward refits: stress book must COVER the worst realized window. Static book Kupiec is diagnostic only.
def refit_recon_beta(cutoff):
    f_tr = FACTOR_RAW_ALIGNED.loc[:cutoff].values.astype(np.float64)
    r_tr = returns_aligned.loc[:cutoff, asset_columns].values.astype(np.float64)
    aug = np.concatenate([np.ones((len(f_tr), 1)), f_tr], axis=1)
    ridge = np.eye(aug.shape[1]) * float(FACTOR_RIDGE_ALPHA)
    ridge[0, 0] = 0.0
    beta = np.linalg.solve(aug.T @ aug + ridge, aug.T @ r_tr)
    resid = r_tr - aug @ beta
    return beta.astype(np.float32), resid.astype(np.float32)

def _reconstruct_local(factors, beta_c, resid_c, multipliers, seed):
    aug = np.concatenate([np.ones((*factors.shape[:-1], 1), dtype=np.float32), factors], axis=-1)
    mean_recon = np.matmul(aug, beta_c)
    rng = np.random.default_rng(seed)
    n_windows = len(resid_c) - WINDOW_SIZE + 1
    starts = rng.choice(n_windows, size=len(mean_recon), replace=True)
    resid_windows = np.stack([resid_c[s:s + WINDOW_SIZE] for s in starts])
    scale = np.sqrt(multipliers)[:, None, None] if multipliers is not None else 1.0
    out = mean_recon + resid_windows * float(RESIDUAL_BOOTSTRAP_SCALE) * scale
    if multipliers is not None:
        out = apply_catastrophe_sleeve_to_returns(out, multipliers=multipliers, seed=seed)
    return np.clip(out, -0.80, 1.50)

def walk_forward_refit_check(cutoff_text, seed_offset):
    cutoff = pd.Timestamp(cutoff_text)
    factor_pre = FACTOR_RAW_ALIGNED.loc[:cutoff]
    if len(factor_pre) < 500:
        return {"cutoff": cutoff_text, "ok": False, "note": "insufficient_pre_cutoff_history"}
    engine_c = FhsFactorEngine(factor_pre, regime_series.loc[factor_pre.index],
                               lam=FHS_EWMA_LAMBDA, block_length=FHS_BLOCK_LENGTH,
                               min_pool=FHS_MIN_REGIME_POOL, seed=SEED + seed_offset)
    beta_c, resid_c = refit_recon_beta(cutoff)
    if len(resid_c) - WINDOW_SIZE + 1 < 32:
        return {"cutoff": cutoff_text, "ok": False, "note": "insufficient_residual_windows"}

    f_crisis, pool_note = engine_c.simulate(WALK_FORWARD_REFIT_SCENARIOS, WINDOW_SIZE, TARGET_REGIME, seed=SEED + seed_offset + 1)
    f_stress, wf_mult = stress_ladder(f_crisis, seed=SEED + seed_offset + 11)
    stress_ret = _reconstruct_local(f_stress, beta_c, resid_c, wf_mult, SEED + seed_offset + 2)
    stress_port = np.clip(stress_ret @ EW_WEIGHTS, -0.95, 1.50)
    stress_terminal = np.prod(1 + stress_port, axis=1) - 1
    stress_q01 = float(np.quantile(stress_terminal, 0.01))

    post = returns_aligned.loc[returns_aligned.index > cutoff, asset_columns].iloc[:WALK_FORWARD_EVAL_DAYS]
    if len(post) < 60:
        return {"cutoff": cutoff_text, "ok": False, "note": "insufficient_post_cutoff_history"}
    post_port = np.clip(post.values.astype(np.float64) @ EW_WEIGHTS, -0.95, 1.50)
    stride = max(1, int(EVAL_WINDOW_STRIDE))
    actual_terminals = [float(np.prod(1 + post_port[s:s + WINDOW_SIZE]) - 1)
                        for s in range(0, len(post_port) - WINDOW_SIZE + 1, stride)]
    actual_min = float(np.min(actual_terminals)) if actual_terminals else np.nan
    covered = bool(np.isfinite(actual_min) and stress_q01 <= actual_min)
    return {
        "cutoff": cutoff_text, "pool_note": pool_note,
        "stress_q01_terminal": stress_q01, "actual_min_terminal": actual_min,
        "stress_covers": covered,
        "ok": bool(covered), "note": "static_unconditional_kupiec_removed_from_stress_gate",
    }

walk_forward_refit_rows = []
for offset, cutoff_text in enumerate([c.strip() for c in WALK_FORWARD_CUTOFFS.split(",") if c.strip()]):
    walk_forward_refit_rows.append(walk_forward_refit_check(cutoff_text, seed_offset=1000 + offset * 100))
walk_forward_refit_df = pd.DataFrame(walk_forward_refit_rows)
display(walk_forward_refit_df)

var5_backtest = conditional_var_backtests.get("var5", {})
var1_backtest = conditional_var_backtests.get("var1", {})
EXTRA_GATES = {
    "conditional_var5_kupiec_ok": bool(var5_backtest.get("kupiec", {}).get("ok", False)),
    "conditional_var5_independence_ok": bool(var5_backtest.get("christoffersen_independence", {}).get("ok", False)),
    "conditional_var5_conditional_coverage_ok": bool(var5_backtest.get("christoffersen_conditional_coverage", {}).get("ok", False)),
    "conditional_var1_kupiec_ok_diagnostic": bool(var1_backtest.get("kupiec", {}).get("ok", False)),
    "walk_forward_stress_covers_all": bool(len(walk_forward_refit_df) > 0 and walk_forward_refit_df["stress_covers"].fillna(False).all()) if "stress_covers" in walk_forward_refit_df else False,
}
print("EXTRA_GATES (V9.7):", json.dumps(EXTRA_GATES, indent=2))

metrics["static_kupiec_uncond_var5_diagnostic"] = kupiec_uncond_var5
metrics["static_kupiec_uncond_var1_diagnostic"] = kupiec_uncond_var1
metrics["static_kupiec_crisis_var5_diagnostic"] = kupiec_crisis_var5
metrics["static_kupiec_crisis_var1_diagnostic"] = kupiec_crisis_var1
metrics["conditional_var_backtests"] = conditional_var_backtests
metrics["conditional_var5_kupiec"] = var5_backtest.get("kupiec")
metrics["conditional_var5_christoffersen_independence"] = var5_backtest.get("christoffersen_independence")
metrics["conditional_var5_christoffersen_conditional_coverage"] = var5_backtest.get("christoffersen_conditional_coverage")
metrics["hill_uncond_synthetic"] = float(hill_uncond) if np.isfinite(hill_uncond) else None
metrics["hill_valid_all"] = float(hill_valid_all) if np.isfinite(hill_valid_all) else None
metrics["hill_uncond_rel_err"] = float(hill_uncond_rel_err) if np.isfinite(hill_uncond_rel_err) else None
metrics["hill_crisis_diagnostic"] = {"synthetic": float(hill_crisis) if np.isfinite(hill_crisis) else None,
                                     "real": float(hill_valid_crisis) if np.isfinite(hill_valid_crisis) else None}
metrics["per_asset_q05_median_rel_err"] = per_asset_q05_median_rel
metrics["walk_forward_refits"] = walk_forward_refit_rows
metrics["cholesky_alpha_selected"] = float(CONFIG.get("cholesky_alpha_selected", CHOLESKY_CALIBRATION_ALPHA))
CONFIG["v97_gate_design"] = "stress_endpoint_by_walk_forward_coverage__daily_var_by_conditional_rolling_kupiec_christoffersen__static_books_diagnostic"
CONFIG["v92_gate_design"] = CONFIG["v97_gate_design"]
CONFIG["conditional_var_backtest"] = conditional_var_backtests
CONFIG["v9_extra_gates"] = {k: bool(v) for k, v in EXTRA_GATES.items()}


In [ ]:
#@title Tail contributors, diagnostic plots, and endpoint gate
var5 = np.quantile(syn_port, 0.05)
cvar5 = syn_port[syn_port <= var5].mean()
tail_mask = syn_port <= var5
asset_terminal = np.prod(1 + to_numpy(synthetic_returns), axis=1) - 1
contrib = pd.Series(asset_terminal[tail_mask].mean(axis=0) * weights, index=train_ds.columns).sort_values()
display(contrib.head(20).to_frame("weighted_tail_contribution"))

fig, axes = plt.subplots(2, 3, figsize=(18, 9))

axes[0, 0].hist(real_port, bins=50, alpha=0.45, label=eval_reference_name)
axes[0, 0].hist(syn_port, bins=50, alpha=0.45, label="fhs v9.7 stress")
axes[0, 0].hist(base_port, bins=50, alpha=0.30, label="fhs v9.7 base")
axes[0, 0].axvline(var5, color="red", linestyle="--", label="Stress VaR 5%")
axes[0, 0].axvline(cvar5, color="black", linestyle=":", label="Stress CVaR 5%")
axes[0, 0].set_title("Terminal equal-weight portfolio returns")
axes[0, 0].legend(fontsize=8)

real_corr = np.nan_to_num(np.corrcoef(to_numpy(real_eval_returns).reshape(-1, N_ASSETS_ACTUAL).T), nan=0.0)
syn_corr = np.nan_to_num(np.corrcoef(to_numpy(synthetic_returns).reshape(-1, N_ASSETS_ACTUAL).T), nan=0.0)
im = axes[0, 1].imshow(syn_corr - real_corr, vmin=-0.5, vmax=0.5, cmap="coolwarm")
axes[0, 1].set_title("Stress synthetic minus eval-reference correlation")
plt.colorbar(im, ax=axes[0, 1], shrink=0.75)

q = np.linspace(0.01, 0.99, 99)
axes[0, 2].plot(np.quantile(real_port, q), np.quantile(syn_port, q), marker=".", linestyle="none", label="stress")
axes[0, 2].plot(np.quantile(real_port, q), np.quantile(base_port, q), marker=".", linestyle="none", alpha=0.45, label="base")
lims = [min(np.quantile(real_port, 0.01), np.quantile(syn_port, 0.01), np.quantile(base_port, 0.01)), max(np.quantile(real_port, 0.99), np.quantile(syn_port, 0.99), np.quantile(base_port, 0.99))]
axes[0, 2].plot(lims, lims, color="black", linewidth=1)
axes[0, 2].set_title("QQ: eval reference vs FHS factor books")
axes[0, 2].legend()
axes[0, 2].grid(True, alpha=0.25)

for i in range(min(25, len(syn_daily))):
    axes[1, 0].plot(np.cumprod(1 + syn_daily[i]) - 1, alpha=0.22)
axes[1, 0].set_title("Stress sample portfolio paths")
axes[1, 0].grid(True, alpha=0.25)

axes[1, 1].hist(stress_multipliers, bins=np.unique(stress_multipliers).size)
axes[1, 1].set_title("Stress multiplier mixture")
axes[1, 1].grid(True, alpha=0.25)

eig_real = np.sort(np.linalg.eigvalsh(real_corr))[-20:]
eig_syn = np.sort(np.linalg.eigvalsh(syn_corr))[-20:]
axes[1, 2].plot(eig_real, label="real top eigen")
axes[1, 2].plot(eig_syn, label="synthetic top eigen")
axes[1, 2].set_title("Correlation eigenstructure")
axes[1, 2].legend()
axes[1, 2].grid(True, alpha=0.25)

plt.tight_layout()
plt.show()

cvar_ref_col = eval_reference_name
# V9 has no neural training loop; the skipped-batch gate is trivially satisfied.
total_batches_seen = 0
total_skipped_batches = 0
skipped_batch_rate = 0.0

normal_score_metrics = metrics_table.loc[normal_candidate_model].to_dict()
stress_score_metrics = metrics_table.loc[stress_candidate_model].to_dict()
scorecard = {
    "product_candidate_is_stress_book": bool(candidate_model == stress_candidate_model),
    "normal_book_beats_gaussian_mmd_multi_diagnostic": bool(metrics_table.loc[normal_candidate_model, "mmd_rbf_projected_multi_mean"] < metrics_table.loc["gaussian_cov_train", "mmd_rbf_projected_multi_mean"]),
    "stress_book_beats_gaussian_mmd_multi": bool(metrics_table.loc[stress_candidate_model, "mmd_rbf_projected_multi_mean"] < metrics_table.loc["gaussian_cov_train", "mmd_rbf_projected_multi_mean"]),
    "beats_gaussian_mmd": bool(metrics_table.loc[candidate_model, "mmd_rbf_projected"] < metrics_table.loc["gaussian_cov_train", "mmd_rbf_projected"]),
    "beats_gaussian_mmd_multi": bool(metrics_table.loc[candidate_model, "mmd_rbf_projected_multi_mean"] < metrics_table.loc["gaussian_cov_train", "mmd_rbf_projected_multi_mean"]),
    "mmd_ratio_within_research_gate": bool(metrics.get("mmd_multi_ratio_candidate_vs_gaussian", np.inf) <= MMD_RATIO_MAX_RESEARCH),
    "beats_gaussian_corr": bool(metrics_table.loc[candidate_model, "corr_mae"] < metrics_table.loc["gaussian_cov_train", "corr_mae"]),
    "beats_t_copula_mmd": bool(metrics_table.loc[candidate_model, "mmd_rbf_projected_multi_mean"] < metrics_table.loc["t_copula_train", "mmd_rbf_projected_multi_mean"]),
    "beats_fhs_mmd": bool(metrics_table.loc[candidate_model, "mmd_rbf_projected_multi_mean"] < metrics_table.loc["filtered_historical_simulation_train", "mmd_rbf_projected_multi_mean"]),
    "beats_same_stack_gaussian_mmd": bool(metrics_table.loc[candidate_model, "mmd_rbf_projected_multi_mean"] < metrics_table.loc["gaussian_factor_same_calibration_stack", "mmd_rbf_projected_multi_mean"]),
    "ties_fhs_mmd_statistically": bool(
        abs(metrics_table.loc[candidate_model, "mmd_rbf_projected_multi_mean"] - metrics_table.loc["filtered_historical_simulation_train", "mmd_rbf_projected_multi_mean"])
        <= TIE_SIGMA * np.sqrt(metrics_table.loc[candidate_model, "mmd_rbf_projected_multi_std"] ** 2 + metrics_table.loc["filtered_historical_simulation_train", "mmd_rbf_projected_multi_std"] ** 2)
    ),
    "ties_same_stack_gaussian_mmd_statistically": bool(
        abs(metrics_table.loc[candidate_model, "mmd_rbf_projected_multi_mean"] - metrics_table.loc["gaussian_factor_same_calibration_stack", "mmd_rbf_projected_multi_mean"])
        <= TIE_SIGMA * np.sqrt(metrics_table.loc[candidate_model, "mmd_rbf_projected_multi_std"] ** 2 + metrics_table.loc["gaussian_factor_same_calibration_stack", "mmd_rbf_projected_multi_std"] ** 2)
    ),
    "tail_cvar_closer_than_same_stack_gaussian": bool(
        abs(regime_calibration.loc["cvar5", candidate_model] - regime_calibration.loc["cvar5", cvar_ref_col])
        < abs(metrics_table.loc["gaussian_factor_same_calibration_stack", "cvar5"] - regime_calibration.loc["cvar5", cvar_ref_col])
    ),
    "corr_near_gaussian": bool(metrics_table.loc[candidate_model, "corr_mae"] <= metrics_table.loc["gaussian_cov_train", "corr_mae"] + CORR_MAE_NEAR_GAUSSIAN_TOL),
    "corr_fidelity_ge_0_80": bool(metrics_table.loc[candidate_model, "correlation_fidelity"] >= 0.80),
    "target_cvar_close_to_eval_reference": bool(abs(regime_calibration.loc["cvar5", candidate_model] - regime_calibration.loc["cvar5", cvar_ref_col]) <= max(0.01, abs(regime_calibration.loc["cvar5", cvar_ref_col]) * 0.35)),
    "stress_walk_forward_1pct_covers_all": bool(EXTRA_GATES.get("walk_forward_stress_covers_all", False)),
    "walk_forward_1pct_covers_all": bool(EXTRA_GATES.get("walk_forward_stress_covers_all", False)),
    "factor_stress_mmd_no_worse_than_base": bool(metrics.get("factor_mmd_stress_multi_mean", np.inf) <= metrics.get("factor_mmd_base_multi_mean", np.inf) * 1.10),
    "stress_stratified_sampling": bool(CONFIG.get("stress_stratified_sampling_applied", False)),
    "endpoint_scenario_count_ok": bool(metrics.get("endpoint_scenario_count_ok", False)),
    "skipped_batch_rate_ok": bool(skipped_batch_rate <= 0.001),
    "no_validation_used_for_guidance_or_cholesky": bool(not CONFIG.get("guidance_uses_validation", True) and not CONFIG.get("cholesky_uses_validation", True)),
    "no_full_window_stress_floor": bool(CONFIG.get("stress_full_window_floor_applied") is False and CONFIG.get("stress_catastrophe_is_full_window_floor") is False),
    "non_overlapping_eval_windows": bool(CONFIG.get("eval_nonoverlap_gate_ok", False)),
    "eval_metric_sample_size_ok": bool(CONFIG.get("eval_metric_sample_size_ok", False)),
}
scorecard["research_champion"] = bool(
    scorecard["product_candidate_is_stress_book"]
    and scorecard["mmd_ratio_within_research_gate"]
    and scorecard["beats_gaussian_mmd_multi"]
    and scorecard["beats_t_copula_mmd"]
    and (scorecard["beats_fhs_mmd"] or (scorecard["ties_fhs_mmd_statistically"] and scorecard["tail_cvar_closer_than_same_stack_gaussian"]))
    and (scorecard["beats_same_stack_gaussian_mmd"] or (scorecard["ties_same_stack_gaussian_mmd_statistically"] and scorecard["tail_cvar_closer_than_same_stack_gaussian"]))
    and scorecard["corr_near_gaussian"]
    and scorecard["corr_fidelity_ge_0_80"]
    and scorecard["no_full_window_stress_floor"]
    and scorecard["factor_stress_mmd_no_worse_than_base"]
    and scorecard["non_overlapping_eval_windows"]
    and scorecard["target_cvar_close_to_eval_reference"]
    and scorecard["walk_forward_1pct_covers_all"]
    and scorecard["stress_stratified_sampling"]
    and scorecard["endpoint_scenario_count_ok"]
    and scorecard["skipped_batch_rate_ok"]
    and scorecard["no_validation_used_for_guidance_or_cholesky"]
)
scorecard.update({k: bool(v) for k, v in EXTRA_GATES.items()})
scorecard["ready_for_stress_endpoint"] = bool(
    scorecard["product_candidate_is_stress_book"]
    and scorecard["beats_gaussian_mmd_multi"]
    and scorecard["beats_gaussian_corr"]
    and scorecard["corr_fidelity_ge_0_80"]
    and scorecard["target_cvar_close_to_eval_reference"]
    and scorecard["walk_forward_1pct_covers_all"]
    and scorecard["stress_stratified_sampling"]
    and scorecard["endpoint_scenario_count_ok"]
    and scorecard["skipped_batch_rate_ok"]
    and scorecard["no_validation_used_for_guidance_or_cholesky"]
)
scorecard["ready_for_conditional_var_endpoint"] = bool(
    scorecard.get("conditional_var5_kupiec_ok", False)
    and scorecard.get("conditional_var5_independence_ok", False)
    and scorecard.get("conditional_var5_conditional_coverage_ok", False)
)
scorecard["static_unconditional_book_diagnostic_only"] = True
scorecard["research_champion"] = bool(
    scorecard["research_champion"]
    and scorecard.get("walk_forward_stress_covers_all", False)
    and scorecard["ready_for_conditional_var_endpoint"]
)
scorecard["ready_for_endpoint"] = bool(scorecard["ready_for_stress_endpoint"])
metrics["skipped_batch_rate"] = float(skipped_batch_rate)
metrics["total_skipped_batches"] = int(total_skipped_batches)
metrics["total_batches_seen"] = int(total_batches_seen)
metrics["research_champion"] = bool(scorecard["research_champion"])
metrics["ready_for_endpoint"] = bool(scorecard["ready_for_endpoint"])
metrics["normal_book_metrics"] = normal_score_metrics
metrics["stress_book_metrics"] = stress_score_metrics
metrics["scorecard"] = scorecard
CONFIG["research_champion"] = bool(scorecard["research_champion"])
CONFIG["ready_for_endpoint"] = bool(scorecard["ready_for_endpoint"])
display(pd.Series(scorecard).to_frame("pass"))
print("Skipped batch rate:", skipped_batch_rate, "| skipped:", total_skipped_batches, "/", total_batches_seen)


## 9. Save Artifacts to Drive

The checkpoint is already saved during training. This cell writes:
- config JSON;
- metrics JSON;
- synthetic scenarios as compressed NumPy;
- tail contribution CSV;
- training history CSV.
        


In [ ]:
#@title Save artifacts
run_id = time.strftime("%Y%m%d_%H%M%S")
run_dir = ARTIFACT_DIR / f"fhs_v9_7_run_{run_id}"
run_dir.mkdir(parents=True, exist_ok=True)

with open(run_dir / "config.json", "w") as f:
    json.dump(CONFIG, f, indent=2)
with open(run_dir / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

synthetic_factor_norm = (np.asarray(synthetic_factor_raw, dtype=np.float32) - as_numpy_local(FACTOR_MU_T)[None, None, :]) / as_numpy_local(FACTOR_SIGMA_T)[None, None, :]

scenario_payload = dict(
    synthetic_returns=to_numpy(synthetic_returns),
    synthetic_returns_base=to_numpy(synthetic_returns_base),
    synthetic_factor_raw=synthetic_factor_raw,
    synthetic_factor_norm=to_numpy(synthetic_factor_norm),
    stress_multipliers=stress_multipliers,
    columns=np.array(train_ds.columns),
    factor_columns=np.array(FACTOR_COLUMNS),
    target_regime=np.array([TARGET_REGIME]),
    selected_guidance_scale=np.array([CONFIG.get("selected_guidance_scale", 1.0)]),
    eval_reference_name=np.array([CONFIG.get("eval_reference_name", "unknown")]),
    ready_for_endpoint=np.array([bool(CONFIG.get("ready_for_endpoint", False))]),
)
if "synthetic_factor_raw_uncalibrated" in globals():
    scenario_payload["synthetic_factor_raw_uncalibrated"] = synthetic_factor_raw_uncalibrated
if "synthetic_returns_base_uncalibrated" in globals():
    scenario_payload["synthetic_returns_base_uncalibrated"] = to_numpy(synthetic_returns_base_uncalibrated)
if "synthetic_returns_uncalibrated" in globals():
    scenario_payload["synthetic_returns_uncalibrated"] = to_numpy(synthetic_returns_uncalibrated)
np.savez_compressed(run_dir / "synthetic_scenarios.npz", **scenario_payload)

factor_bank_path = None
if EXPORT_FACTOR_SCENARIO_BANK:
    bank_dtype = np.float16 if FACTOR_BANK_DTYPE == "float16" else np.float32
    factor_bank_payload = {
        "factor_paths_base": np.asarray(synthetic_factor_raw, dtype=bank_dtype),
        "factor_paths_stress": np.asarray(stress_ladder(np.asarray(synthetic_factor_raw, dtype=np.float32), seed=SEED + 202)[0], dtype=bank_dtype),
        "stress_multipliers": np.asarray(stress_multipliers, dtype=np.float32),
        "factor_columns": np.array(FACTOR_COLUMNS),
        "window_size": np.array([WINDOW_SIZE]),
        "target_regime": np.array([TARGET_REGIME]),
        "run_id": np.array([run_id]),
        "shock_mode": np.array([STRESS_SHOCK_MODE]),
    }
    factor_bank_path = run_dir / "factor_scenario_bank_fp16.npz"
    np.savez_compressed(factor_bank_path, **factor_bank_payload)
# V9: no neural training loop, so there is no training history artifact.
metrics_table.to_csv(run_dir / "metrics_table.csv")
regime_calibration.to_csv(run_dir / "regime_calibration.csv")
if "walk_forward_df" in globals() and len(walk_forward_df):
    walk_forward_df.to_csv(run_dir / "walk_forward_crisis_audit.csv", index=False)
if "guidance_sweep_df" in globals() and len(guidance_sweep_df):
    guidance_sweep_df.to_csv(run_dir / "guidance_sweep.csv", index=False)
if "mmd_stability_table" in globals():
    mmd_stability_table.to_csv(run_dir / "mmd_stability_table.csv")
if "calibration_audit_table" in globals():
    calibration_audit_table.to_csv(run_dir / "calibration_audit_table.csv")
if "synthetic_subsample_sensitivity" in globals():
    synthetic_subsample_sensitivity.to_csv(run_dir / "synthetic_subsample_sensitivity.csv", index=False)
if "portfolio_suite_table" in globals():
    portfolio_suite_table.to_csv(run_dir / "portfolio_suite_table.csv", index=False)
if "endpoint_contract_table" in globals():
    endpoint_contract_table.to_csv(run_dir / "endpoint_contract.csv")
    with open(run_dir / "endpoint_contract.json", "w") as f:
        json.dump(endpoint_contract, f, indent=2)
if "factor_space_metrics" in globals():
    factor_space_metrics.to_frame("value").to_csv(run_dir / "factor_space_metrics.csv")
contrib.to_csv(run_dir / "tail_contributors.csv")

factor_package = {
    "factor_columns": FACTOR_COLUMNS,
    "sector_names": sector_names,
    "pca_explained_variance_ratio_sum": CONFIG.get("pca_explained_variance_ratio_sum"),
    "recon_beta_shape": list(RECON_BETA.shape),
    "residual_bootstrap_scale": RESIDUAL_BOOTSTRAP_SCALE,
    "stress_mix_weights": STRESS_MIX_WEIGHTS,
    "stress_mix_multipliers": STRESS_MIX_MULTIPLIERS,
    "factor_calibration_applied": bool(CONFIG.get("factor_calibration_applied", False)),
    "factor_calibration_alpha": float(CONFIG.get("factor_calibration_alpha", 0.0)),
    "factor_calibration_shrinkage": float(CONFIG.get("factor_calibration_shrinkage", 0.0)),
    "factor_calibration_target_source": CONFIG.get("factor_calibration_target_source", "unknown"),
    "cholesky_calibration_applied": bool(CONFIG.get("cholesky_calibration_applied", False)),
    "cholesky_calibration_alpha": float(CONFIG.get("cholesky_calibration_alpha", 0.0)),
    "cholesky_shrinkage": float(CONFIG.get("cholesky_shrinkage", 0.0)),
    "cholesky_calibration_target_source": CONFIG.get("cholesky_calibration_target_source", "unknown"),
    "stress_shock_mode": CONFIG.get("stress_shock_mode_applied", STRESS_SHOCK_MODE),
    "stress_location_shift_applied": bool(CONFIG.get("stress_location_shift_applied", False)),
    "stress_location_shift_alpha": STRESS_LOCATION_SHIFT_ALPHA,
    "stress_catastrophe_sleeve_applied": bool(CONFIG.get("stress_catastrophe_sleeve_applied", False)),
    "stress_catastrophe_weight": STRESS_CATASTROPHE_WEIGHT,
    "stress_catastrophe_daily_shock": CONFIG.get("stress_catastrophe_daily_shock"),
    "daily_var5_safety_uncond_applied": bool(CONFIG.get("daily_var5_safety_uncond_applied", False)),
    "daily_var5_safety_crisis_base_applied": bool(CONFIG.get("daily_var5_safety_crisis_base_applied", False)),
    "daily_var5_safety_uncond_method": CONFIG.get("daily_var5_safety_uncond_method", "unknown"),
    "daily_var5_safety_uncond_uses_validation": bool(CONFIG.get("daily_var5_safety_uncond_uses_validation", True)),
    "daily_var5_safety_uncond_implied_multiplier": CONFIG.get("daily_var5_safety_uncond_implied_multiplier"),
    "daily_var5_safety_crisis_base_method": CONFIG.get("daily_var5_safety_crisis_base_method", "unknown"),
    "daily_var5_safety_crisis_base_uses_validation": bool(CONFIG.get("daily_var5_safety_crisis_base_uses_validation", True)),
    "daily_var5_safety_crisis_base_implied_multiplier": CONFIG.get("daily_var5_safety_crisis_base_implied_multiplier"),
    "factor_tail_location_anchor_applied": bool(CONFIG.get("factor_tail_location_anchor_applied", False)),
    "stress_full_window_floor_applied": bool(CONFIG.get("stress_full_window_floor_applied", False)),
    "survivorship_disclosure": SURVIVORSHIP_DISCLOSURE,
}
with open(run_dir / "factor_package.json", "w") as f:
    json.dump(factor_package, f, indent=2)

api_manifest = {
    "endpoint": "/api/v1/workspaces/{workspaceId}/market-simulation",
    "model_family": MODEL_FAMILY,
    "notebook_version": "v9.7",
    "v92_gate_design": CONFIG.get("v92_gate_design", "unknown"),
    "pit_universe_active": bool(CONFIG.get("pit_universe_active", False)),
    "pit_member_day_raw_coverage": CONFIG.get("pit_member_day_raw_coverage"),
    "pit_member_day_raw_coverage_basis": CONFIG.get("pit_member_day_raw_coverage_basis"),
    "pit_member_days_raw_total": CONFIG.get("pit_member_days_raw_total"),
    "pit_member_days_raw_covered": CONFIG.get("pit_member_days_raw_covered"),
    "pit_member_day_coverage": CONFIG.get("pit_member_day_coverage"),
    "pit_member_day_coverage_basis": CONFIG.get("pit_member_day_coverage_basis"),
    "survivorship_disclosure": CONFIG.get("survivorship_disclosure", "unknown"),
    "cholesky_alpha_selected": float(CONFIG.get("cholesky_alpha_selected", CHOLESKY_CALIBRATION_ALPHA)),
    "candidate_model": CONFIG.get("candidate_model", "fhs_v9_stress"),
    "research_champion": bool(CONFIG.get("research_champion", False)),
    "ready_for_endpoint": bool(CONFIG.get("ready_for_endpoint", False)),
    "endpoint_gate": "deploy stress scenario bank when ready_for_stress_endpoint is true; validate daily VaR through conditional rolling Kupiec and Christoffersen; static scenario books remain diagnostic",
    "checkpoint_path": None,  # V9 is a CPU statistical engine: no neural checkpoint
    "artifact_dir": str(run_dir),
    "columns_path": str(run_dir / "synthetic_scenarios.npz"),
    "config_path": str(run_dir / "config.json"),
    "metrics_path": str(run_dir / "metrics.json"),
    "factor_package_path": str(run_dir / "factor_package.json"),
    "endpoint_contract_path": str(run_dir / "endpoint_contract.json"),
    "factor_scenario_bank_path": str(factor_bank_path) if factor_bank_path else None,
    "endpoint_serving_strategy": "static_factor_bank_projection",
    "selected_guidance_scale": float(CONFIG.get("selected_guidance_scale", 1.0)),
    "target_regime": int(TARGET_REGIME),
    "n_assets_actual": int(N_ASSETS_ACTUAL),
    "n_factors_actual": int(N_FACTORS_ACTUAL),
    "n_macro_features": int(N_MACRO_FEATURES),
    "eval_reference_name": CONFIG.get("eval_reference_name", "unknown"),
    "validation_protocol": CONFIG.get("validation_protocol", "unknown"),
    "guidance_uses_validation": bool(CONFIG.get("guidance_uses_validation", True)),
    "cholesky_uses_validation": bool(CONFIG.get("cholesky_uses_validation", True)),
    "stress_scenario_mode_applied": bool(CONFIG.get("stress_scenario_mode_applied", False)),
    "stress_stratified_sampling_applied": bool(CONFIG.get("stress_stratified_sampling_applied", False)),
    "stress_multiplier_counts": CONFIG.get("stress_multiplier_counts", {}),
    "stress_model_for_walk_forward": CONFIG.get("stress_model_for_walk_forward", CONFIG.get("stress_candidate_model", "fhs_v9_stress")),
    "skipped_batch_rate": float(metrics.get("skipped_batch_rate", np.nan)),
    "mmd_multi_ratio_candidate_vs_gaussian": float(metrics.get("mmd_multi_ratio_candidate_vs_gaussian", np.nan)),
    "eval_uses_synthetic_subsample": bool(CONFIG.get("eval_uses_synthetic_subsample", False)),
    "primary_eval_subsample_seed": int(CONFIG.get("primary_eval_subsample_seed", -1)),
    "subsample_base_mmd_multi_mean_mean": float(metrics.get("subsample_base_mmd_multi_mean_mean", np.nan)),
    "subsample_base_mmd_multi_mean_std": float(metrics.get("subsample_base_mmd_multi_mean_std", np.nan)),
    "endpoint_contract_status": CONFIG.get("endpoint_contract_status", "unknown"),
    "endpoint_default_scenarios": int(CONFIG.get("endpoint_default_scenarios", -1)),
    "endpoint_min_scenarios": int(CONFIG.get("endpoint_min_scenarios", -1)),
    "endpoint_min_stress_scenarios": int(CONFIG.get("endpoint_min_stress_scenarios", -1)),
    "endpoint_stress_book_q01": float(metrics.get("endpoint_stress_book_q01", np.nan)),
    "factor_calibration_applied": bool(CONFIG.get("factor_calibration_applied", False)),
    "factor_calibration_uses_validation": bool(CONFIG.get("factor_calibration_uses_validation", True)),
    "factor_calibration_target_source": CONFIG.get("factor_calibration_target_source", "unknown"),
    "cholesky_calibration_applied": bool(CONFIG.get("cholesky_calibration_applied", False)),
    "cholesky_calibration_target_source": CONFIG.get("cholesky_calibration_target_source", "unknown"),
}
with open(run_dir / "blsprime_market_simulation_manifest.json", "w") as f:
    json.dump(api_manifest, f, indent=2)

print("Saved artifacts to:", run_dir)
print("No neural checkpoint: V9 is a CPU statistical engine (factor bank is the deployable artifact).")
print("API manifest:", run_dir / "blsprime_market_simulation_manifest.json")
print("Research champion:", api_manifest["research_champion"])
print("Ready for endpoint:", api_manifest["ready_for_endpoint"])


## 10. Promotion rule (V9.7)

Promote the FHS factor stress candidate to the endpoint only when the full scorecard passes:
it must beat Gaussian-cov, t-copula, naive FHS, and the same-stack Gaussian ablation on
multi-seed MMD; stay within correlation tolerance; pass Kupiec VaR5/VaR1 exception tests
on validation days; match the Hill tail index within tolerance; keep per-asset q05 error
within tolerance; and cover all three walk-forward refit cutoffs. Export the factor
scenario bank regardless: the endpoint serves the bank of whichever engine is champion.


## V9.7 interpretation rule

If the same-stack Gaussian ablation matches the FHS stress candidate on the gated metrics, the
dynamic-vol and regime machinery is not earning its complexity: ship the simpler engine.
If naive FHS matches the factor FHS stress candidate, the factor structure is not earning its
complexity. Complexity must buy measurable tail accuracy or it does not ship.
